# NASA C-MAPSS: Remaining Useful Life (RUL) Prediction
End-to-end, leakage-safe pipeline for turbofan engine degradation (FD001, FD002, FD003, FD004).

Change `DATASET_ID` in section 1.1 and run all cells from top to bottom.

**Notebook map**

| # | Section | # | Section |
|---|---|---|---|
| 01 | Environment setup | 10 | XGBoost regression |
| 02 | Data ingestion | 11 | Feature engineering for deep learning |
| 03 | Pre-EDA domain ETL | 12 | LSTM |
| 04 | Engine-level split, imputation, RUL labels | 13 | Hybrid CNN-LSTM |
| 05 | EDA: conditions, variance, outliers | 14 | Temporal Convolutional Network |
| 06 | Leakage-safe preprocessing | 15 | Temporal Transformer |
| 07 | EDA on normalized data | 16 | Model comparison and selection |
| 08 | Feature engineering for traditional ML | 17 | Final training and test evaluation |
| 09 | Random Forest regression | 18 | Final project verification |


# 01. Environment Setup and Imports

## 1.1 Install Packages
Run once, then restart the kernel.

In [187]:
# Install the required packages once, then restart the kernel.
%pip install numpy pandas scikit-learn scipy torch xgboost plotly joblib nbformat
%pip install python-dotenv kagglehub sqlalchemy pymysql

## 1.2 Prerequisite Imports
Standard library, third-party packages.

In [188]:
import math
import json
import os
import platform
import random
import sys
from dotenv import load_dotenv
from pathlib import Path

import joblib
import kagglehub
import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from torch.utils.data import DataLoader, TensorDataset

try:
    import xgboost as xgb
except ImportError as error:
    raise ImportError("XGBoost is required for section 10. Install it with: pip install xgboost") from error

## 1.3 Project Configuration
Every tunable value lives in this one cell. `QUICK_RUN = True` shrinks epochs and search sizes for a fast smoke test.

In [189]:
# ----- dataset and paths -----
load_dotenv()                    # read the .env file
DATASET_ID = "FD002"             # FD001 | FD002 | FD003 | FD004
DATA_SOURCE = "kaggle_api"            # local | kaggle_api | mysql
ARTIFACT_DIR = os.getenv("CMAPSS_OUT_DIR", "artifacts")

# ----- target and windowing -----
RUL_CAP = 125                    # piecewise-linear cap for training targets (cycles)
WINDOW_SIZE = 30                 # sliding-window length (cycles)
VALIDATION_SIZE = 0.2            # fraction of training ENGINES held out for validation
VALIDATION_TAIL = RUL_CAP        # validation rows with true RUL below this value form the test-like "tail"

# ----- training -----
RANDOM_STATE = 42
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6
DEVICE = "cpu"                   # "cpu" always works; "cuda" is used only if available

# ----- preprocessing -----
VARIANCE_THRESHOLD = 1e-4        # a sensor is flat inside a condition if its std is below this
MIN_UNIQUE_VALUES = 3            # ... or if it has at most this many distinct values
MIN_CLUSTER_ROWS = 50            # conditions with fewer training rows use the global scaler
EXPECTED_CONDITIONS = {"FD001": 1, "FD002": 6, "FD003": 1, "FD004": 6}

# ----- hyperparameter search -----
CV_FOLDS = 3                     # GroupKFold folds (grouped by engine) for RF and XGBoost
RF_SEARCH_ITER = 8
XGB_SEARCH_ITER = 8
TUNE_EPOCHS = 12                 # epochs per neural-network tuning trial
RF_GRID = {"n_estimators": [150, 250], "max_depth": [12, 20, None],
           "min_samples_leaf": [3, 5, 10], "max_features": [0.3, 0.5, "sqrt"]}
XGB_GRID = {"n_estimators": [200, 400], "learning_rate": [0.03, 0.06, 0.1],
            "max_depth": [3, 5, 7], "subsample": [0.7, 0.9], "colsample_bytree": [0.6, 0.9]}
# first entry of each list is the default configuration; the rest are tuning candidates
DL_CONFIGS = {
    "LSTM": [{"hidden": 64, "layers": 2, "dropout": 0.2},
             {"hidden": 32, "layers": 1, "dropout": 0.0},
             {"hidden": 96, "layers": 2, "dropout": 0.3}],
    "CNN-LSTM": [{"filters": 32, "kernel": 3, "hidden": 64, "layers": 1, "dropout": 0.2},
                 {"filters": 16, "kernel": 5, "hidden": 32, "layers": 1, "dropout": 0.1},
                 {"filters": 48, "kernel": 3, "hidden": 96, "layers": 2, "dropout": 0.3}],
    "TCN": [{"channels": [32, 32, 64], "kernel": 3, "dropout": 0.2},
            {"channels": [16, 32], "kernel": 3, "dropout": 0.1},
            {"channels": [32, 64, 64, 64], "kernel": 2, "dropout": 0.2}],
    "Transformer": [{"d_model": 32, "nhead": 4, "layers": 2, "ff": 64, "dropout": 0.1},
                    {"d_model": 16, "nhead": 2, "layers": 1, "ff": 32, "dropout": 0.0},
                    {"d_model": 64, "nhead": 4, "layers": 2, "ff": 128, "dropout": 0.2}],
}

# ----- smoke-test switch -----
QUICK_RUN = False                # True: tiny epochs/searches, only the default network configuration
if QUICK_RUN:
    EPOCHS, TUNE_EPOCHS, EARLY_STOPPING_PATIENCE = 3, 2, 2
    CV_FOLDS, RF_SEARCH_ITER, XGB_SEARCH_ITER = 2, 2, 2
    DL_CONFIGS = {name: configs[:2] for name, configs in DL_CONFIGS.items()}

# ----- column names (the raw files have no header) -----
# operating settings: altitude (ft), Mach number, throttle resolver angle
SETTING_COLS = ["alt_ft", "Mach", "TRA"]
# sensors, in the column order of the raw files (sensor 1 to sensor 21):
# T2/T24/T30/T50 temperatures, P2/P15/P30 pressures, Nf/Nc fan and core speeds, epr pressure ratio,
# Ps30 static pressure, phi fuel-flow ratio, NRf/NRc corrected speeds, BPR bypass ratio, farB fuel-air ratio,
# htBleed bleed enthalpy, Nf_dmd/PCNfR_dmd demanded speeds, W31/W32 coolant bleeds
SENSOR_COLS = ["T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30",
               "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32"]
RAW_COLS = ["unit", "cycle"] + SETTING_COLS + SENSOR_COLS

## 1.4 Environment Verification
Checks library versions, the compute device, the artifact folder, and that the configuration values make sense.

In [190]:
# validate configuration values early so mistakes fail with a clear message
assert DATASET_ID in EXPECTED_CONDITIONS, f"DATASET_ID must be one of {list(EXPECTED_CONDITIONS)}, got {DATASET_ID!r}"
assert WINDOW_SIZE >= 2, "WINDOW_SIZE must be at least 2"
assert 0.0 < VALIDATION_SIZE < 1.0, "VALIDATION_SIZE must be between 0 and 1"
assert RUL_CAP > 0 and BATCH_SIZE > 0 and EPOCHS > 0 and CV_FOLDS >= 2

# fall back to CPU when CUDA was requested but is not available
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("CUDA requested but not available, using CPU")
    DEVICE = "cpu"
DEVICE = torch.device(DEVICE)

# create the artifact folder once so every later save can assume it exists
ARTIFACT_PATH = Path(ARTIFACT_DIR)
ARTIFACT_PATH.mkdir(parents=True, exist_ok=True)

print("python :", sys.version.split()[0], "|", platform.system())
print("numpy  :", np.__version__, "| pandas:", pd.__version__)
print("torch  :", torch.__version__, "| xgboost:", xgb.__version__, "| plotly:", plotly.__version__)
print("device :", DEVICE, "| dataset:", DATASET_ID)
print("artifacts folder:", ARTIFACT_PATH.resolve())

python : 3.13.18 | Linux
numpy  : 2.4.6 | pandas: 2.2.3
torch  : 2.14.1+cpu | xgboost: 3.4.1 | plotly: 5.24.1
device : cpu | dataset: FD002
artifacts folder: /content/artifacts


## 1.5 Reproducibility
Seeds for Python, NumPy, and PyTorch. `set_seed` is called again before every model so each run is repeatable on its own.

In [191]:
def set_seed(seed):
    """Seed Python, NumPy, and PyTorch so results are repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)
# ask PyTorch for deterministic kernels where they exist (warn instead of crash otherwise)
torch.use_deterministic_algorithms(True, warn_only=True)
print("seed set to", RANDOM_STATE)

seed set to 42


## 1.6 Reusable Operations

**NASA score convention:** `error = predicted - actual`. Early predictions (`error < 0`) cost `exp(-error/13) - 1`; late predictions (`error >= 0`) cost `exp(error/10) - 1`. Late is penalized harder because a missed failure is worse than an early inspection.

In [192]:
def rmse(y_true, y_pred):
    """Root mean squared error."""
    difference = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    return float(np.sqrt(np.mean(difference ** 2)))


def nasa_score(y_true, y_pred):
    """Sum of the NASA asymmetric penalties (error = predicted - actual)."""
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    penalty = np.where(error < 0, np.exp(-error / 13.0) - 1.0, np.exp(error / 10.0) - 1.0)
    return float(penalty.sum())


def evaluate_predictions(y_true, y_pred):
    """Return RMSE, MAE, the summed NASA score, and the NASA score per sample."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    assert y_true.shape == y_pred.shape, "targets and predictions must have the same shape"
    assert np.isfinite(y_pred).all(), "predictions contain NaN or inf"
    score = nasa_score(y_true, y_pred)
    return {"rmse": rmse(y_true, y_pred), "mae": float(mean_absolute_error(y_true, y_pred)),
            "nasa": score, "nasa_mean": score / len(y_true)}


def add_rul_labels(frame, rul_cap):
    """Add RUL_linear (max cycle - cycle) and RUL (linear value capped at rul_cap) from the engine's own cycles."""
    labeled = frame.copy()
    last_cycle = labeled.groupby("unit")["cycle"].transform("max")
    labeled["RUL_linear"] = (last_cycle - labeled["cycle"]).astype("int32")
    labeled["RUL"] = labeled["RUL_linear"].clip(upper=rul_cap).astype("float32")
    return labeled


def assign_conditions(frame, settings_scaler, kmeans_model):
    """Operating-condition label per row: nearest fitted KMeans center, or 0 when the dataset has one condition."""
    if kmeans_model is None:
        return np.zeros(len(frame), dtype="int32")
    scaled_settings = settings_scaler.transform(frame[SETTING_COLS])
    return kmeans_model.predict(scaled_settings).astype("int32")


def normalize_sensors(frame, live_sensors, condition_scalers, fallback_scaler):
    """Standardize live sensors with the scaler of each row's condition (global fallback for rare conditions)."""
    normalized = frame.copy()
    normalized[live_sensors] = normalized[live_sensors].astype("float64")
    condition_values = frame["condition"].to_numpy()
    for condition in np.unique(condition_values):
        mask = condition_values == condition
        scaler = condition_scalers.get(int(condition), fallback_scaler)
        normalized.loc[mask, live_sensors] = scaler.transform(frame.loc[mask, live_sensors])
    return normalized


def build_windows(frame, feature_cols, window_size, last_only=False):
    """Sliding windows built for each engine separately (windows never cross engines).

    Every cycle gets one window that ends at that cycle. Engines shorter than the window are
    left-padded by repeating their first observation. With last_only=True only the final
    window of each engine is returned. Output windows have shape (n_windows, window_size, n_features).
    """
    assert frame.sort_values(["unit", "cycle"]).index.equals(frame.index), "frame must be sorted by unit and cycle"
    window_list, unit_list, cycle_list = [], [], []
    for unit_id, engine in frame.groupby("unit", sort=True):
        values = engine[feature_cols].to_numpy(dtype=np.float32)               # (n_cycles, n_features)
        padding = np.repeat(values[:1], window_size - 1, axis=0)               # (window_size - 1, n_features)
        padded = np.concatenate([padding, values], axis=0)                     # (n_cycles + window_size - 1, n_features)
        engine_windows = sliding_window_view(padded, window_size, axis=0)      # (n_cycles, n_features, window_size)
        engine_windows = engine_windows.transpose(0, 2, 1)                     # (n_cycles, window_size, n_features)
        engine_cycles = engine["cycle"].to_numpy()
        if last_only:
            engine_windows, engine_cycles = engine_windows[-1:], engine_cycles[-1:]
        window_list.append(np.ascontiguousarray(engine_windows))
        unit_list.append(np.full(len(engine_cycles), unit_id))
        cycle_list.append(engine_cycles)
    return np.concatenate(window_list), np.concatenate(unit_list), np.concatenate(cycle_list)


def extract_window_features(windows, extra_values, extra_names, sensor_names):
    """Tabular features from windows of shape (n, window_size, n_sensors); only past and current cycles are used."""
    steps = windows.shape[1]
    time_centered = np.arange(steps, dtype=np.float64) - (steps - 1) / 2.0
    slope = (windows * time_centered[None, :, None]).sum(axis=1) / (time_centered ** 2).sum()
    blocks = {"last": windows[:, -1, :], "mean": windows.mean(axis=1), "std": windows.std(axis=1),
              "min": windows.min(axis=1), "max": windows.max(axis=1),
              "change": windows[:, -1, :] - windows[:, 0, :], "slope": slope}
    names = [f"{sensor}_{stat}" for stat in blocks for sensor in sensor_names] + list(extra_names)
    matrix = np.concatenate(list(blocks.values()) + [np.asarray(extra_values, dtype=np.float64)], axis=1)
    return matrix.astype(np.float32), names


def predict_network(model, inputs, batch_size=512):
    """Batched inference in eval mode without gradients; returns RUL in cycles (never below 0)."""
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(inputs), batch_size):
            batch = inputs[start:start + batch_size].to(DEVICE)
            outputs.append(model(batch).cpu())
    return np.clip(torch.cat(outputs).numpy() * RUL_CAP, 0.0, None)  # network predicts RUL / RUL_CAP


def train_network(model, optimizer, loss_function, train_loader, epochs, patience=None, x_val=None, y_val=None, verbose=True):
    """Train with optional epoch-level validation and early stopping.

    With validation data the best epoch (lowest validation RMSE) is restored at the end. Without it the
    model trains for exactly `epochs` epochs. Targets are scaled by RUL_CAP so the loss stays near 0..1.
    """
    model.to(DEVICE)
    history = {"epoch": [], "train_loss": [], "val_rmse": []}
    best_rmse, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), (batch_y / RUL_CAP).to(DEVICE)
            optimizer.zero_grad()
            loss = loss_function(model(batch_x), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # keeps recurrent gradients from exploding
            optimizer.step()
            running_loss += loss.item() * len(batch_x)
        history["epoch"].append(epoch)
        history["train_loss"].append(running_loss / len(train_loader.dataset))
        if x_val is None:
            history["val_rmse"].append(float("nan"))
            continue
        val_rmse = rmse(y_val, predict_network(model, x_val))
        assert math.isfinite(val_rmse), f"validation RMSE is not finite at epoch {epoch}"
        history["val_rmse"].append(val_rmse)
        if val_rmse < best_rmse - 1e-6:
            best_rmse, best_epoch, wait = val_rmse, epoch, 0
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            wait += 1
        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"epoch {epoch:3d}/{epochs} | train loss {history['train_loss'][-1]:.5f} | val-tail RMSE {val_rmse:.2f}")
        if patience is not None and wait >= patience:
            if verbose:
                print(f"early stopping at epoch {epoch} (best epoch {best_epoch})")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    history["best_rmse"], history["best_epoch"] = best_rmse, best_epoch
    return history


print("helper functions defined")

helper functions defined


### Sanity check of the NASA score
Hand-computed values: an early error of -10 gives `exp(10/13) - 1`, a late error of +10 gives `exp(10/10) - 1`, and a perfect prediction gives 0.

In [193]:
# a late prediction must cost more than an early one of the same size
assert abs(nasa_score([50], [40]) - (math.exp(10 / 13) - 1)) < 1e-9
assert abs(nasa_score([50], [60]) - (math.exp(10 / 10) - 1)) < 1e-9
assert nasa_score([50], [50]) == 0.0
assert nasa_score([50], [60]) > nasa_score([50], [40])
# the score of several engines is the sum of the individual penalties
assert abs(nasa_score([50, 80], [40, 90]) - (nasa_score([50], [40]) + nasa_score([80], [90]))) < 1e-9
print("early -10:", round(nasa_score([50], [40]), 4), "| late +10:", round(nasa_score([50], [60]), 4))

early -10: 1.1581 | late +10: 1.7183


# 02. Data Ingestion


## 2.1 Dataset File Discovery
Finds the three files of the selected dataset anywhere below `DATA_DIR`. Only `DATASET_ID` decides what is loaded.

In [194]:
def read_txt_files(folder):
    """Read train, test, and RUL files of DATASET_ID from a folder (subfolders are searched too)."""
    frames = []
    for prefix in ("train", "test", "RUL"):
        matches = sorted(Path(folder).rglob(f"{prefix}_{DATASET_ID}.txt"))
        if not matches:
            raise FileNotFoundError(f"{prefix}_{DATASET_ID}.txt not found under {folder}")
        frames.append(pd.read_csv(matches[0], sep=r"\s+", header=None))
    return frames

if DATA_SOURCE == "local":
    train_raw_df, test_raw_df, test_rul_df = read_txt_files(os.environ["CMAPSS_DATA_DIR"])

elif DATA_SOURCE == "kaggle_api":
    kaggle_folder = kagglehub.dataset_download(os.environ["KAGGLE_DATASET_HANDLE"])  # cached after the first download
    train_raw_df, test_raw_df, test_rul_df = read_txt_files(kaggle_folder)

elif DATA_SOURCE == "mysql":
    table = os.environ["MYSQL_TABLE"]
    assert table.isidentifier(), "MYSQL_TABLE must be a plain table name"
    url = URL.create("mysql+pymysql", username=os.environ["MYSQL_USER"], password=os.getenv("MYSQL_PASSWORD") or None,
                     host=os.environ["MYSQL_HOST"], port=int(os.environ["MYSQL_PORT"]), database=os.environ["MYSQL_DATABASE"])
    engine = create_engine(url)
    query = text(f"SELECT * FROM {table} WHERE dataset_id = :dataset_id AND split = :split ORDER BY unit, cycle")

    def read_split(split):
        return pd.read_sql(query, engine, params={"dataset_id": DATASET_ID, "split": split})

    train_raw_df = read_split("train")[RAW_COLS]     # same 26 columns as the txt files
    test_raw_df = read_split("test")[RAW_COLS]
    test_rul_df = read_split("rul")[["rul"]]         # one row per test engine, ordered by engine id

else:
    raise ValueError(f"DATA_SOURCE must be local, kaggle_api, or mysql, got {DATA_SOURCE!r}")

test_rul_df.columns = ["true_rul"]                   # the notebook expects this column name
print(f"[{DATA_SOURCE}] train {train_raw_df.shape} | test {test_raw_df.shape} | RUL labels {test_rul_df.shape}")

Using Colab cache for faster access to the 'nasa-cmaps' dataset.
[kaggle_api] train (53759, 26) | test (33991, 26) | RUL labels (259, 1)


## 2.2 Training Data

In [195]:
print("train raw shape:", train_raw_df.shape)

train raw shape: (53759, 26)


## 2.3 Test Data

In [196]:
print("test raw shape:", test_raw_df.shape)

test raw shape: (33991, 26)


## 2.4 Test RUL Labels
Row *i* of this file is the true RUL of test engine *i* at its last recorded cycle. These labels are used only for the final evaluation.

In [197]:
print("test RUL labels shape:", test_rul_df.shape)

test RUL labels shape: (259, 1)


## 2.5 Assign Column Names
26 columns: engine id, cycle, 3 operating settings, 21 sensors. A wrong column count means the wrong file or a broken download, so it stops here.

In [198]:
for name, frame in (("train", train_raw_df), ("test", test_raw_df)):
    if frame.shape[1] != len(RAW_COLS):
        raise ValueError(f"{name} file has {frame.shape[1]} columns, expected {len(RAW_COLS)}")
train_raw_df.columns = RAW_COLS
test_raw_df.columns = RAW_COLS
print(list(train_raw_df.columns))

['unit', 'cycle', 'alt_ft', 'Mach', 'TRA', 'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']


## 2.6 Initial Data Inspection

In [199]:
for name, frame in (("train", train_raw_df), ("test", test_raw_df)):
    cycles_per_engine = frame.groupby("unit")["cycle"].max()
    print(f"[{name}] rows={len(frame):,} | engines={frame['unit'].nunique()} | "
          f"cycles per engine min/mean/max = {cycles_per_engine.min()}/{cycles_per_engine.mean():.0f}/{cycles_per_engine.max()}")
print("test RUL labels:", len(test_rul_df))
print(train_raw_df.dtypes.value_counts().to_string())
train_raw_df.head()

[train] rows=53,759 | engines=260 | cycles per engine min/mean/max = 128/207/378
[test] rows=33,991 | engines=259 | cycles per engine min/mean/max = 21/131/367
test RUL labels: 259
float64    22
int64       4


,unit,cycle,alt_ft,Mach,TRA,T2,T24,T30,T50,P2,...,phi,NRf,NRc,BPR,farB,htBleed,Nf_dmd,PCNfR_dmd,W31,W32
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,...,183.06,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,...,130.42,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,...,164.22,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,...,130.72,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,...,164.31,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286


## 2.7 Initial Data Validation
Schema checks with meaningful errors.

In [200]:
if len(train_raw_df) == 0 or len(test_raw_df) == 0 or len(test_rul_df) == 0:
    raise ValueError("at least one of the loaded files is empty")
non_numeric = [c for c in RAW_COLS if not pd.api.types.is_numeric_dtype(train_raw_df[c]) or not pd.api.types.is_numeric_dtype(test_raw_df[c])]
if non_numeric:
    raise ValueError(f"non-numeric columns found: {non_numeric}")
if test_raw_df["unit"].nunique() != len(test_rul_df):
    raise ValueError(f"{test_raw_df['unit'].nunique()} test engines but {len(test_rul_df)} RUL labels")
print("schema check passed")

schema check passed


# 03. Pre-EDA Domain-Specific ETL
Structural cleaning and integrity checks.


## 3.1 Data Profiling

In [201]:
profile = pd.DataFrame({
    "rows": [len(train_raw_df), len(test_raw_df)],
    "engines": [train_raw_df["unit"].nunique(), test_raw_df["unit"].nunique()],
    "missing values": [int(train_raw_df.isna().sum().sum()), int(test_raw_df.isna().sum().sum())],
    "duplicate rows": [int(train_raw_df.duplicated().sum()), int(test_raw_df.duplicated().sum())],
    "duplicate (unit, cycle)": [int(train_raw_df.duplicated(["unit", "cycle"]).sum()), int(test_raw_df.duplicated(["unit", "cycle"]).sum())],
}, index=["train", "test"])
profile

,rows,engines,missing values,duplicate rows,"duplicate (unit, cycle)"
train,53759,260,0,0,0
test,33991,259,0,0,0


Descriptive statistics of the operating settings and sensors (training file). Columns with a tiny spread are candidates for removal in section 5.3.

In [202]:
train_raw_df[SETTING_COLS + SENSOR_COLS].describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
alt_ft,53759.0,23.9984,14.7474,0.0000,10.0046,25.0013,41.9980,42.0080
Mach,53759.0,0.5721,0.3100,0.0000,0.2507,0.7000,0.8400,0.8420
TRA,53759.0,94.0460,14.2377,60.0000,100.0000,100.0000,100.0000,100.0000
T2,53759.0,472.9102,26.3897,445.0000,445.0000,462.5400,491.1900,518.6700
T24,53759.0,579.6724,37.2894,535.5300,549.5700,555.9800,607.3400,644.5200
T30,53759.0,1419.9710,105.9463,1243.7300,1352.7600,1369.1800,1499.3700,1612.8800
T50,53759.0,1205.4420,119.1234,1023.7700,1123.6550,1138.8900,1306.8500,1439.2300
P2,53759.0,8.0320,3.6138,3.9100,3.9100,7.0500,10.5200,14.6200
P15,53759.0,11.6007,5.4318,5.7100,5.7200,9.0300,15.4900,21.6100
P30,53759.0,282.6068,146.0053,136.8000,139.9350,194.6600,394.0800,555.8200


## 3.2 Data Cleaning
Types first, then ordering, then duplicates. Rows are never dropped just because they look extreme: late-life readings are the degradation signal.

In [203]:
# integer ids keep memory small, floats for every measurement
train_df = train_raw_df.copy()
test_df = test_raw_df.copy()
for frame in (train_df, test_df):
    frame[["unit", "cycle"]] = frame[["unit", "cycle"]].astype("int32")
    frame[SETTING_COLS + SENSOR_COLS] = frame[SETTING_COLS + SENSOR_COLS].astype("float64")
print(train_df.dtypes.value_counts().to_string())

float64    24
int32       2


Chronological order inside each engine is required for windows and RUL labels, so rows are sorted by `(unit, cycle)`.

In [204]:
# chronological order inside each engine is required for windows and RUL labels
train_df = train_df.sort_values(["unit", "cycle"]).reset_index(drop=True)
test_df = test_df.sort_values(["unit", "cycle"]).reset_index(drop=True)
print("sorted by (unit, cycle)")

sorted by (unit, cycle)


Exact duplicate rows are removed. If two rows share (unit, cycle) but differ in values, the data is ambiguous and the notebook stops instead of guessing.

In [205]:
rows_before = (len(train_df), len(test_df))
train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df = test_df.drop_duplicates().reset_index(drop=True)
for name, frame in (("train", train_df), ("test", test_df)):
    if frame.duplicated(["unit", "cycle"]).any():
        raise ValueError(f"{name}: conflicting duplicate (unit, cycle) records")
print("rows removed (train, test):", rows_before[0] - len(train_df), rows_before[1] - len(test_df))

rows removed (train, test): 0 0


Invalid numbers (`inf`) become missing values so that one rule handles both. Missing values are filled in section 4.3 with statistics learned from the training partition only.

In [206]:
measurement_cols = SETTING_COLS + SENSOR_COLS
infinite_count = int(np.isinf(train_df[measurement_cols]).sum().sum() + np.isinf(test_df[measurement_cols]).sum().sum())
train_df[measurement_cols] = train_df[measurement_cols].replace([np.inf, -np.inf], np.nan)
test_df[measurement_cols] = test_df[measurement_cols].replace([np.inf, -np.inf], np.nan)
print("infinite values converted to NaN:", infinite_count)
print("missing values now (train, test):", int(train_df.isna().sum().sum()), int(test_df.isna().sum().sum()))

infinite values converted to NaN: 0
missing values now (train, test): 0 0


## 3.3 Data Integrity
Hard checks. The notebook stops if one fails.

In [207]:
def cycles_are_contiguous(frame):
    """True when every engine has cycles 1..n without gaps."""
    grouped = frame.groupby("unit")["cycle"]
    return bool(((grouped.min() == 1) & (grouped.max() == grouped.count())).all())


train_units = np.sort(train_df["unit"].unique())
test_units = np.sort(test_df["unit"].unique())
integrity = {
    "train engine ids are 1..N": bool((train_units == np.arange(1, len(train_units) + 1)).all()),
    "test engine ids are 1..N": bool((test_units == np.arange(1, len(test_units) + 1)).all()),
    "cycles start at 1, no gaps (train)": cycles_are_contiguous(train_df),
    "cycles start at 1, no gaps (test)": cycles_are_contiguous(test_df),
    "unique (unit, cycle) (train)": not train_df.duplicated(["unit", "cycle"]).any(),
    "unique (unit, cycle) (test)": not test_df.duplicated(["unit", "cycle"]).any(),
    "chronological order (train)": bool(train_df.sort_values(["unit", "cycle"]).index.equals(train_df.index)),
    "chronological order (test)": bool(test_df.sort_values(["unit", "cycle"]).index.equals(test_df.index)),
    "numeric measurements": all(pd.api.types.is_numeric_dtype(train_df[c]) for c in measurement_cols),
    "train and test columns match": list(train_df.columns) == list(test_df.columns),
    "one RUL label per test engine": len(test_rul_df) == len(test_units),
    "RUL labels are non-negative": bool((test_rul_df["true_rul"] >= 0).all()),
}
integrity_report = pd.Series(integrity, name="passed").to_frame()
assert integrity_report["passed"].all(), f"integrity check failed:\n{integrity_report[~integrity_report['passed']]}"
integrity_report

,passed
train engine ids are 1..N,True
test engine ids are 1..N,True
"cycles start at 1, no gaps (train)",True
"cycles start at 1, no gaps (test)",True
"unique (unit, cycle) (train)",True
"unique (unit, cycle) (test)",True
chronological order (train),True
chronological order (test),True
numeric measurements,True
train and test columns match,True


Test engine ids are exactly 1..N and sorted, so row *i* of the RUL file belongs to test engine *i*. That alignment is used again in section 17.

## 3.4 Initial Data Overview
Engine lifetimes: training engines run to failure, test engines are cut off early, which is why the RUL file exists.

In [208]:
train_lifetimes = train_df.groupby("unit")["cycle"].max()
test_lifetimes = test_df.groupby("unit")["cycle"].max()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Cycles recorded per engine", "True RUL of test engines at their last cycle"))
fig.add_trace(go.Histogram(x=train_lifetimes, nbinsx=20, name="train (run to failure)", opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=test_lifetimes, nbinsx=20, name="test (truncated)", opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=test_rul_df["true_rul"], nbinsx=20, name="test true RUL", marker_color="seagreen"), row=1, col=2)
fig.update_xaxes(title_text="cycles", row=1, col=1); fig.update_xaxes(title_text="RUL (cycles)", row=1, col=2)
fig.update_yaxes(title_text="engines")
fig.update_layout(barmode="overlay", height=380, width=1000, title_text=f"{DATASET_ID}: engine lifetimes and test RUL labels")
fig.show()
print("engines shorter than WINDOW_SIZE (train, test):", int((train_lifetimes < WINDOW_SIZE).sum()), int((test_lifetimes < WINDOW_SIZE).sum()))

engines shorter than WINDOW_SIZE (train, test): 0 6


# 04. Engine-Level Split, Imputation, and RUL Labels


## 4.1 Engine-Level Train/Validation Split
`GroupShuffleSplit` with `groups = unit` splits whole engines, never rows. Rows of one engine are strongly correlated, so a row-wise split would leak the future of an engine into training. The test set is untouched.

In [209]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE)
train_index, val_index = next(splitter.split(train_df, groups=train_df["unit"]))
train_part_df = train_df.iloc[train_index].reset_index(drop=True)
val_part_df = train_df.iloc[val_index].reset_index(drop=True)
print(f"train partition: {train_part_df['unit'].nunique()} engines, {len(train_part_df):,} rows")
print(f"validation partition: {val_part_df['unit'].nunique()} engines, {len(val_part_df):,} rows")

train partition: 208 engines, 43,464 rows
validation partition: 52 engines, 10,295 rows


## 4.2 Split Verification

In [210]:
train_part_units = set(train_part_df["unit"].unique())
val_part_units = set(val_part_df["unit"].unique())
assert train_part_units.isdisjoint(val_part_units), "an engine appears in both partitions"
assert train_part_units | val_part_units == set(train_df["unit"].unique()), "some engines were lost"
assert len(train_part_units) > 0 and len(val_part_units) > 0, "empty partition"
assert train_part_df.sort_values(["unit", "cycle"]).index.equals(train_part_df.index), "train partition lost its order"
assert val_part_df.sort_values(["unit", "cycle"]).index.equals(val_part_df.index), "validation partition lost its order"
print("engine-level split verified: no engine is in both partitions")

engine-level split verified: no engine is in both partitions


## 4.3 Training-Fitted Imputation
The median of each measurement is learned on the **training partition only** and applied to validation and test. The C-MAPSS files have no missing values, so here this is a safe no-op; it keeps the pipeline correct if data with gaps is ever used.

In [211]:
imputer = SimpleImputer(strategy="median")
imputer.fit(train_part_df[measurement_cols])
print("imputer fitted on", len(train_part_df), "training rows")

imputer fitted on 43464 training rows


Apply the training-fitted imputer to the three frames and confirm that nothing is missing afterwards.

In [212]:
train_part_df[measurement_cols] = imputer.transform(train_part_df[measurement_cols])
val_part_df[measurement_cols] = imputer.transform(val_part_df[measurement_cols])
test_df[measurement_cols] = imputer.transform(test_df[measurement_cols])
assert not train_part_df[measurement_cols].isna().any().any() and not val_part_df[measurement_cols].isna().any().any() \
    and not test_df[measurement_cols].isna().any().any(), "NaN values remain after imputation"
print("no missing values remain in train, validation, or test")

no missing values remain in train, validation, or test


## 4.4 RUL Label Construction
`RUL_linear = last cycle of the engine - current cycle`, and the training target is `RUL = min(RUL_linear, RUL_CAP)`.
Labels come from each engine's **own** recorded life. Training and validation engines run to failure, so this is valid for them. Test engines get **no** such label: their truth comes only from the RUL file in section 17.

In [213]:
train_part_df = add_rul_labels(train_part_df, RUL_CAP)
val_part_df = add_rul_labels(val_part_df, RUL_CAP)
assert (train_part_df.groupby("unit")["RUL_linear"].min() == 0).all(), "every training engine must end at RUL 0"
assert "RUL" not in test_df.columns and "RUL_linear" not in test_df.columns, "test frame must not carry RUL labels"
print("RUL range (train partition):", train_part_df["RUL"].min(), "-", train_part_df["RUL"].max())
train_part_df[["unit", "cycle", "RUL_linear", "RUL"]].head(164)

RUL range (train partition): 0.0 - 125.0


,unit,cycle,RUL_linear,RUL
0,1,1,148,125.0
1,1,2,147,125.0
2,1,3,146,125.0
3,1,4,145,125.0
4,1,5,144,125.0
...,...,...,...,...
159,2,11,258,125.0
160,2,12,257,125.0
161,2,13,256,125.0
162,2,14,255,125.0


# 05. Exploratory Data Analysis (Training Partition Only)
Every plot and decision in this section uses `train_part_df` only. Validation and test engines never influence a choice.


## 5.1 Univariate Analysis
Distributions of the operating settings and sensors. Several spikes or multi-modal shapes hint at flat sensors or several operating conditions.

In [214]:
eda_cols = SETTING_COLS + SENSOR_COLS
univariate_stats = train_part_df[eda_cols].agg(["mean", "std", "min", "max", "skew"]).T
univariate_stats.round(4)

,mean,std,min,max,skew
alt_ft,23.9920,14.7508,0.0000,42.0080,-0.2586
Mach,0.5718,0.3102,0.0000,0.8420,-0.8322
TRA,94.0641,14.2200,60.0000,100.0000,-1.9782
T2,472.9059,26.3932,445.0000,518.6700,0.4550
T24,579.6868,37.2829,535.5300,644.5200,0.4378
T30,1420.0792,105.9107,1243.7300,1612.8800,0.1648
T50,1205.5933,119.1264,1030.0800,1439.2300,0.4283
P2,8.0331,3.6143,3.9100,14.6200,0.5359
P15,11.6031,5.4323,5.7100,21.6100,0.6218
P30,282.6876,146.0123,136.8000,555.8200,0.6959


Histogram of every setting and sensor.

In [215]:
# one histogram per setting and sensor to see spikes (flat sensors) and several bumps (operating conditions)

fig = make_subplots(rows=6, cols=4, subplot_titles=eda_cols, vertical_spacing=0.04)
palette = fig.layout.template.layout.colorway

for i, column in enumerate(eda_cols):
    x = train_part_df[column].dropna().to_numpy(dtype=float)
    r, c = i // 4 + 1, i % 4 + 1
    dark = palette[i % len(palette)]
    fill = '#' + ''.join(f'{int(0.25 * int(dark[j:j+2], 16) + 191.25):02X}' for j in (1, 3, 5))

    fig.add_trace(go.Histogram(x=x, nbinsx=30, histnorm="probability density",
        marker=dict(color=fill, line=dict(color=dark, width=1.2)), showlegend=False), row=r, col=c)

    if len(np.unique(x)) > 1:
        xx = np.linspace(x.min(), x.max(), 300)
        fig.add_trace(go.Scatter(x=xx, y=gaussian_kde(x)(xx), line=dict(color=dark, width=2), showlegend=False), row=r, col=c)

fig.update_layout(height=1100, width=1000, title_text=f"{DATASET_ID}: distributions of settings and sensors (training partition)")
fig.show()

Output hidden; open in https://colab.research.google.com to view.

Box plots of the z-scored variables.

In [216]:
# box plots on z-scored columns so all 24 variables share one axis; constant columns would divide by zero, so they are skipped
spread = train_part_df[eda_cols].std()
varying_cols = [c for c in eda_cols if spread[c] > 0]
standardized = (train_part_df[varying_cols] - train_part_df[varying_cols].mean()) / spread[varying_cols]
fig = go.Figure()
for column in varying_cols:
    fig.add_trace(go.Box(y=standardized[column], name=column, boxpoints=False, showlegend=False))
fig.update_layout(height=420, width=1000, yaxis_title="z-score", title_text="Standardized spread of varying columns (outliers hidden here, analyzed in 5.4)")
fig.show()

Output hidden; open in https://colab.research.google.com to view.

Cycle and RUL distributions.

In [217]:
# cycle and RUL distributions: how long engines live and how many rows sit on the capped plateau
fig = make_subplots(rows=1, cols=3, subplot_titles=("cycles per engine", "RUL (linear)", f"RUL (capped at {RUL_CAP})"))
fig.add_trace(go.Histogram(x=train_part_df.groupby("unit")["cycle"].max(), nbinsx=20), row=1, col=1)
fig.add_trace(go.Histogram(x=train_part_df["RUL_linear"], nbinsx=40, marker_color="darkorange"), row=1, col=2)
fig.add_trace(go.Histogram(x=train_part_df["RUL"], nbinsx=40, marker_color="seagreen"), row=1, col=3)
fig.update_layout(showlegend=False, height=350, width=1000)
fig.show()
print(f"{(train_part_df['RUL_linear'] >= RUL_CAP).mean():.1%} of training rows sit on the cap")

40.2% of training rows sit on the cap


## 5.2 Operating Condition Analysis
FD001 and FD003 run at one operating condition, FD002 and FD004 at six. Sensor levels shift between conditions, which would hide degradation if left alone.

The strategy is chosen from the dataset id and **verified against the data**: the number of distinct (rounded) setting combinations in the training partition must match.

In [218]:
n_conditions = EXPECTED_CONDITIONS[DATASET_ID]
rounded_settings = train_part_df[SETTING_COLS].round({"alt_ft": 0, "Mach": 2, "TRA": 0}) + 0.0  # + 0.0 turns -0.0 into 0.0
observed_combinations = int(rounded_settings.drop_duplicates().shape[0])
assert observed_combinations == n_conditions, (
    f"{DATASET_ID} should have {n_conditions} operating conditions but the training settings show {observed_combinations}")
strategy = "global normalization" if n_conditions == 1 else f"KMeans with {n_conditions} clusters, one scaler per cluster"
print(f"{DATASET_ID}: {observed_combinations} operating condition(s) observed -> {strategy}")

FD002: 6 operating condition(s) observed -> KMeans with 6 clusters, one scaler per cluster


Fit the settings scaler and KMeans on the **training partition only**. The settings are standardized first because altitude (tens of thousands of feet) would otherwise dominate Mach number and throttle angle.

In [219]:
if n_conditions == 1:
    settings_scaler, kmeans = None, None
else:
    settings_scaler = StandardScaler().fit(train_part_df[SETTING_COLS])
    kmeans = KMeans(n_clusters=n_conditions, n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(settings_scaler.transform(train_part_df[SETTING_COLS]))
print("clustering model:", "none (single condition)" if kmeans is None else kmeans)

clustering model: KMeans(n_clusters=6, n_init=10, random_state=42)


Assign condition labels to every partition. Validation and test rows go to the nearest **training** cluster center; nothing is refit.

In [220]:
train_part_df["condition"] = assign_conditions(train_part_df, settings_scaler, kmeans)
val_part_df["condition"] = assign_conditions(val_part_df, settings_scaler, kmeans)
test_df["condition"] = assign_conditions(test_df, settings_scaler, kmeans)
print("rows per condition (train):", np.bincount(train_part_df["condition"]).tolist())
print("rows per condition (val)  :", np.bincount(val_part_df["condition"], minlength=n_conditions).tolist())
print("rows per condition (test) :", np.bincount(test_df["condition"], minlength=n_conditions).tolist())

rows per condition (train): [10857, 6572, 6450, 6535, 6507, 6543]
rows per condition (val)  : [2601, 1524, 1552, 1587, 1537, 1494]
rows per condition (test) : [8483, 5107, 5042, 5063, 5148, 5148]


Verification: every training row has a valid label, all conditions are populated, and each distinct setting combination maps to exactly one cluster.

In [221]:
assert set(train_part_df["condition"].unique()) == set(range(n_conditions)), "a training condition is empty or out of range"
assert val_part_df["condition"].between(0, n_conditions - 1).all() and test_df["condition"].between(0, n_conditions - 1).all()
combination_id = rounded_settings.groupby(SETTING_COLS, sort=False).ngroup()
cluster_table = pd.crosstab(combination_id, train_part_df["condition"])
assert ((cluster_table > 0).sum(axis=1) == 1).all(), "KMeans split one setting combination across clusters"
print("each setting combination maps to exactly one cluster")
sample_rows = train_part_df.sample(min(5000, len(train_part_df)), random_state=RANDOM_STATE)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Operating conditions", "Training rows per condition"))
fig.add_trace(go.Scatter(x=sample_rows["alt_ft"], y=sample_rows["Mach"], mode="markers",
                         marker=dict(size=4, color=sample_rows["condition"], colorscale="Turbo"), showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(n_conditions)), y=np.bincount(train_part_df["condition"]), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="alt_ft (altitude)", row=1, col=1); fig.update_yaxes(title_text="Mach (Mach number)", row=1, col=1)
fig.update_xaxes(title_text="condition", dtick=1, row=1, col=2); fig.update_yaxes(title_text="training rows", row=1, col=2)
fig.update_layout(height=380, width=1000)
fig.show()

each setting combination maps to exactly one cluster


How much do sensor levels differ between conditions? Large differences in the means (compared with the spread inside one condition) are the reason for per-condition normalization.

In [222]:
level_by_condition = train_part_df.groupby("condition")[["T24", "T30", "P30", "Ps30"]].mean()
level_by_condition.round(2)

,T24,T30,P30,Ps30
condition,,,,
0,549.71,1354.51,138.63,42.15
1,604.92,1502.05,394.31,45.49
2,536.87,1262.73,175.42,36.80
3,607.57,1485.59,334.49,44.45
4,642.68,1590.49,553.35,47.55
5,555.80,1366.76,194.44,41.99


## 5.3 Zero-Variance and Near-Zero-Variance Analysis
A sensor is **flat** inside a condition if its standard deviation is below `VARIANCE_THRESHOLD` or it has at most `MIN_UNIQUE_VALUES` distinct values. A sensor is removed only when it is flat inside **every** condition: a sensor that only differs *between* conditions still carries information after per-condition normalization.

The decision is learned from the training partition; no sensor name is hard-coded.

In [223]:
within_condition_std = train_part_df.groupby("condition")[SENSOR_COLS].std().fillna(0.0)
within_condition_unique = train_part_df.groupby("condition")[SENSOR_COLS].nunique()
flat_in_condition = (within_condition_std < VARIANCE_THRESHOLD) | (within_condition_unique <= MIN_UNIQUE_VALUES)
flat_sensors = flat_in_condition.columns[flat_in_condition.all(axis=0)].tolist()
live_sensors = [c for c in SENSOR_COLS if c not in flat_sensors]
if len(live_sensors) == 0:
    raise ValueError("every sensor was flagged as flat, check VARIANCE_THRESHOLD and the data")
print("flat sensors removed  :", flat_sensors)
print("live sensors retained :", len(live_sensors), live_sensors)

flat sensors removed  : ['T2', 'P2', 'P15', 'epr', 'farB', 'Nf_dmd', 'PCNfR_dmd']
live sensors retained : 14 ['T24', 'T30', 'T50', 'P30', 'Nf', 'Nc', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'htBleed', 'W31', 'W32']


Visual check: the largest within-condition spread of each sensor (log scale). Red bars are removed.

In [224]:
# max within-condition std per sensor on a log axis; red bars are removed
max_std = within_condition_std.max().clip(lower=1e-8)
fig = go.Figure(go.Bar(x=SENSOR_COLS, y=max_std.values, marker_color=["crimson" if c in flat_sensors else "steelblue" for c in SENSOR_COLS]))
fig.add_hline(y=VARIANCE_THRESHOLD, line_dash="dash", annotation_text="threshold")
fig.update_yaxes(type="log", title_text="max within-condition std")
fig.update_layout(height=380, width=1000, title_text="Sensor spread inside operating conditions (red = removed)")
fig.show()

## 5.4 Outlier Analysis
Z-scores are computed **inside each operating condition** (training statistics), so a normal reading of a different condition is not mistaken for an outlier. Outliers are described, never deleted: turbofan degradation produces extreme late-life values on purpose.

In [225]:
condition_mean = train_part_df.groupby("condition")[live_sensors].transform("mean")
condition_std = train_part_df.groupby("condition")[live_sensors].transform("std").replace(0, np.nan)
condition_z = (train_part_df[live_sensors] - condition_mean) / condition_std
z_outlier = condition_z.abs() > 3
q1, q3 = condition_z.quantile(0.25), condition_z.quantile(0.75)
iqr_outlier = (condition_z < q1 - 1.5 * (q3 - q1)) | (condition_z > q3 + 1.5 * (q3 - q1))
outlier_table = pd.DataFrame({"IQR outliers %": iqr_outlier.mean() * 100, "|z|>3 %": z_outlier.mean() * 100})
outlier_table.sort_values("|z|>3 %", ascending=False).round(2)

,IQR outliers %,|z|>3 %
Nc,6.23,2.26
NRc,5.56,2.16
NRf,4.49,1.57
Nf,4.62,1.56
T30,0.75,0.27
htBleed,1.25,0.27
T24,0.61,0.26
phi,0.75,0.26
P30,0.67,0.26
W32,0.71,0.25


Outlier rate per operating condition: are the unusual readings concentrated in one regime?

In [226]:
# outlier rate by operating condition: are the unusual readings concentrated in one regime?
z_outlier.groupby(train_part_df["condition"]).mean().mul(100).round(2)

,T24,T30,T50,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32
condition,,,,,,,,,,,,,,
0,0.28,0.40,0.28,0.28,1.94,2.28,0.18,0.37,1.94,2.09,0.37,0.20,0.27,0.24
1,0.27,0.26,0.15,0.20,0.06,2.24,0.17,0.18,0.11,2.21,0.17,0.14,0.14,0.26
2,0.34,0.17,0.22,0.40,2.20,2.23,0.14,0.39,2.34,2.26,0.17,0.11,0.20,0.40
3,0.20,0.26,0.18,0.29,2.17,2.17,0.12,0.21,2.23,2.05,0.20,0.49,0.20,0.28
4,0.15,0.22,0.11,0.18,0.43,2.40,0.14,0.20,0.29,2.24,0.28,0.48,0.25,0.12
5,0.31,0.26,0.11,0.17,2.29,2.25,0.14,0.14,2.25,2.15,0.18,0.26,0.20,0.20


Genuine degradation or data-quality problem? If outliers are concentrated in the last cycles of an engine's life, they are degradation. Equal rates early and late would point to noise or errors.

In [227]:
near_failure = (train_part_df["RUL_linear"] <= 30).to_numpy()
rate_near_failure = z_outlier[near_failure].mean() * 100
rate_earlier = z_outlier[~near_failure].mean() * 100
fig = go.Figure()
fig.add_trace(go.Bar(x=live_sensors, y=rate_earlier.values, name="earlier life"))
fig.add_trace(go.Bar(x=live_sensors, y=rate_near_failure.values, name="last 30 cycles"))
fig.update_layout(barmode="group", height=380, width=1000, yaxis_title="|z| > 3 rate (%)",
                  title_text="Outliers are concentrated near failure (degradation), not spread evenly")
fig.show()

# 06. Leakage-Safe Preprocessing
Order of operations: (1) keep the selected sensors, (2) fit one scaler per operating condition on the **training partition only**, (3) apply the fitted scalers to training, validation, and test **without refitting**.


## 6.1 Apply the Sensor Selection
The imputer (4.3), the condition model (5.2), and the sensor selection (5.3) were all fitted on the training partition. The same `live_sensors` list is used for every partition.

In [228]:
base_cols = ["unit", "cycle"] + SETTING_COLS + ["condition"] + live_sensors
train_sel_df = train_part_df[base_cols + ["RUL_linear", "RUL"]].copy()
val_sel_df = val_part_df[base_cols + ["RUL_linear", "RUL"]].copy()
test_sel_df = test_df[base_cols].copy()
print("train", train_sel_df.shape, "| validation", val_sel_df.shape, "| test", test_sel_df.shape)

train (43464, 22) | validation (10295, 22) | test (33991, 20)


## 6.2 Operating-Condition-Based Normalization
FD001 and FD003 have one condition, so this is a single global `StandardScaler`. FD002 and FD004 get one `StandardScaler` per cluster. A cluster with fewer than `MIN_CLUSTER_ROWS` training rows uses a **global fallback scaler**, also fitted on training rows only.

In [229]:
condition_scalers = {}
for condition in sorted(train_sel_df["condition"].unique()):
    condition_rows = train_sel_df.loc[train_sel_df["condition"] == condition, live_sensors]
    if len(condition_rows) >= MIN_CLUSTER_ROWS:
        condition_scalers[int(condition)] = StandardScaler().fit(condition_rows)
fallback_scaler = StandardScaler().fit(train_sel_df[live_sensors])  # used for conditions with too few training rows
print("conditions with their own scaler:", sorted(condition_scalers), "| global fallback fitted on", len(train_sel_df), "rows")

conditions with their own scaler: [0, 1, 2, 3, 4, 5] | global fallback fitted on 43464 rows


Transform the training partition with the fitted scalers.

In [230]:
train_prepared_df = normalize_sensors(train_sel_df, live_sensors, condition_scalers, fallback_scaler)
print("training partition normalized:", train_prepared_df.shape)

training partition normalized: (43464, 22)


Transform the validation partition with the **training** scalers (no refitting).

In [231]:
val_prepared_df = normalize_sensors(val_sel_df, live_sensors, condition_scalers, fallback_scaler)
print("validation partition normalized with training scalers:", val_prepared_df.shape)

validation partition normalized with training scalers: (10295, 22)


Transform the test set with the **training** scalers (no refitting).

In [232]:
test_prepared_df = normalize_sensors(test_sel_df, live_sensors, condition_scalers, fallback_scaler)
print("test set normalized with training scalers:", test_prepared_df.shape)

test set normalized with training scalers: (33991, 20)


## 6.3 Preprocessing Verification

In [233]:
for name, frame in (("train", train_prepared_df), ("validation", val_prepared_df), ("test", test_prepared_df)):
    assert list(frame[base_cols].columns) == base_cols, f"{name}: unexpected columns"
    assert np.isfinite(frame[live_sensors].to_numpy()).all(), f"{name}: non-finite sensor values"
    assert frame["condition"].between(0, n_conditions - 1).all(), f"{name}: invalid condition label"
# every scaler was fitted on exactly the training rows of its condition (nothing from validation or test)
for condition, scaler in condition_scalers.items():
    assert scaler.n_samples_seen_ == int((train_sel_df["condition"] == condition).sum()), "scaler saw non-training rows"
assert fallback_scaler.n_samples_seen_ == len(train_part_df), "fallback scaler saw non-training rows"
# inside each condition the normalized training sensors have mean 0 and std 1
for condition in condition_scalers:
    rows = train_prepared_df.loc[train_prepared_df["condition"] == condition, live_sensors]
    assert np.allclose(rows.mean(), 0, atol=1e-6), "training mean is not 0 after scaling"
print("all preprocessing checks passed | features:", len(live_sensors))

all preprocessing checks passed | features: 14


## 6.4 Save the Preprocessing Components
Everything needed to reproduce the transformations at inference time is stored in one file.

In [234]:
preprocessor = {
    "dataset_id": DATASET_ID, "measurement_cols": measurement_cols, "imputer": imputer,
    "n_conditions": n_conditions, "settings_scaler": settings_scaler, "kmeans": kmeans,
    "live_sensors": live_sensors, "condition_scalers": condition_scalers, "fallback_scaler": fallback_scaler,
    "window_size": WINDOW_SIZE, "rul_cap": RUL_CAP, "base_cols": base_cols,
}
preprocessor_path = ARTIFACT_PATH / f"preprocessor_{DATASET_ID.lower()}.joblib"
joblib.dump(preprocessor, preprocessor_path)
print("saved", preprocessor_path.name)

saved preprocessor_fd002.joblib


# 07. EDA on Normalized Training Data
Back to analysis, now on the condition-normalized training partition, where degradation is visible even for FD002 and FD004.


## 7.1 Bivariate Analysis
**Sensor versus RUL:** Spearman correlation ranks sensors by how monotonically they follow the remaining life.

In [235]:
rho_with_rul = train_prepared_df[live_sensors].corrwith(train_prepared_df["RUL"], method="spearman")
ranked_sensors = rho_with_rul.abs().sort_values(ascending=False).index.tolist()
top_sensors = ranked_sensors[:4]
rho_with_rul.loc[ranked_sensors].round(3).to_frame("Spearman rho vs RUL")

,Spearman rho vs RUL
Ps30,-0.711
T50,-0.681
BPR,-0.673
htBleed,-0.600
T24,-0.587
T30,-0.576
phi,0.502
P30,0.475
W31,0.450
W32,0.448


Scatter plots of the four leading sensors against RUL.

In [236]:
sample_rows = train_prepared_df.sample(min(4000, len(train_prepared_df)), random_state=RANDOM_STATE)
fig = make_subplots(rows=1, cols=4, subplot_titles=top_sensors)
for position, sensor in enumerate(top_sensors):
    fig.add_trace(go.Scatter(x=sample_rows["RUL"], y=sample_rows[sensor], mode="markers", marker=dict(size=3, opacity=0.4), showlegend=False), row=1, col=position + 1)
    fig.update_xaxes(title_text="RUL (capped)", row=1, col=position + 1)
fig.update_layout(height=350, width=1100, title_text="Top sensors versus RUL (normalized)")
fig.show()

**Sensor versus cycle:** the same top sensors across the life of three engines.

In [237]:
fig = make_subplots(rows=1, cols=4, subplot_titles=top_sensors)
for position, sensor in enumerate(top_sensors):
    for unit_id in sorted(train_prepared_df["unit"].unique())[:3]:
        engine = train_prepared_df[train_prepared_df["unit"] == unit_id]
        fig.add_trace(go.Scatter(x=engine["cycle"], y=engine[sensor], mode="lines", name=f"engine {unit_id}", legendgroup=f"engine {unit_id}",
                                 showlegend=(position == 0), line=dict(color=["#1f77b4", "#ff7f0e", "#2ca02c"][int(unit_id) % 3])), row=1, col=position + 1)
    fig.update_xaxes(title_text="cycle", row=1, col=position + 1)
fig.update_layout(height=350, width=1100)
fig.show()

**Sensor versus operating settings:** correlation of each normalized sensor with the three settings. After per-condition normalization the values should be close to zero; large values would mean the condition effect was not fully removed. Settings that never change (FD001, FD003) have no defined correlation and show as 0.

In [238]:
setting_sensor_corr = train_prepared_df[SETTING_COLS + live_sensors].corr().loc[SETTING_COLS, live_sensors].fillna(0.0)
fig = go.Figure(go.Heatmap(z=setting_sensor_corr.values, x=live_sensors, y=SETTING_COLS, zmin=-1, zmax=1, colorscale="RdBu_r", colorbar=dict(title="correlation")))
fig.update_layout(height=300, width=1000, title_text="Settings versus normalized sensors")
fig.show()

**Sensor versus sensor** and **engine-level behavior:** two leading sensors colored by RUL, then the leading sensor for five engines against their remaining life.

In [239]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{top_sensors[0]} vs {top_sensors[1]} (colored by RUL)", f"{top_sensors[0]} against remaining life"))
fig.add_trace(go.Scatter(x=sample_rows[top_sensors[0]], y=sample_rows[top_sensors[1]], mode="markers", showlegend=False,
                         marker=dict(size=4, color=sample_rows["RUL"], colorscale="Viridis", showscale=True, colorbar=dict(title="RUL", x=0.45))), row=1, col=1)
for unit_id in sorted(train_prepared_df["unit"].unique())[:5]:
    engine = train_prepared_df[train_prepared_df["unit"] == unit_id]
    fig.add_trace(go.Scatter(x=engine["RUL_linear"], y=engine[top_sensors[0]], mode="lines", name=f"engine {unit_id}"), row=1, col=2)
fig.update_xaxes(title_text=top_sensors[0], row=1, col=1); fig.update_yaxes(title_text=top_sensors[1], row=1, col=1)
fig.update_xaxes(title_text="RUL (linear, runs toward failure)", autorange="reversed", row=1, col=2); fig.update_yaxes(title_text=top_sensors[0], row=1, col=2)
fig.update_layout(height=400, width=1100)
fig.show()

## 7.2 Correlation Analysis
Spearman correlation matrix of the live sensors, a list of highly correlated pairs, and the variance inflation factor (VIF) as a multicollinearity profile (`VIF = diag(inverse correlation matrix)`, above 10 means strongly redundant).
Nothing is removed: tree models and sequence models tolerate correlated inputs, and correlated sensors often measure the same degrading component.

In [240]:
sensor_corr = train_prepared_df[live_sensors].corr(method="spearman")
fig = go.Figure(go.Heatmap(z=sensor_corr.values, x=live_sensors, y=live_sensors, zmin=-1, zmax=1, colorscale="RdBu_r", colorbar=dict(title="Spearman rho")))
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=650, width=720, title_text="Sensor correlation matrix")
fig.show()

Highly correlated sensor pairs and the VIF profile.

In [241]:
high_pairs = [(a, b, round(float(sensor_corr.loc[a, b]), 3)) for i, a in enumerate(live_sensors) for b in live_sensors[i + 1:]
              if abs(sensor_corr.loc[a, b]) > 0.9]
high_pairs = sorted(high_pairs, key=lambda item: -abs(item[2]))
vif_values = pd.Series(np.diag(np.linalg.pinv(sensor_corr.values)), index=live_sensors).sort_values(ascending=False)
print(f"pairs with |rho| > 0.9: {len(high_pairs)} | strongest: {high_pairs[:5]}")
print("sensors with VIF > 10:", vif_values[vif_values > 10].index.tolist())
vif_values.round(1).to_frame("VIF").head(8)

pairs with |rho| > 0.9: 1 | strongest: [('Nf', 'NRf', 0.909)]
sensors with VIF > 10: []


,VIF
NRf,5.9
Nf,5.9
NRc,4.3
Nc,4.2
Ps30,3.5
T50,2.8
BPR,2.8
htBleed,1.9


**Finding:** the retained sensors are partly redundant (several pairs above 0.9), as expected when sensors watch the same component. All sensors are kept, and the models decide which ones matter (see the feature-importance plots in sections 9 and 10).

## 7.3 Sensor Degradation Analysis
Average normalized sensor value against remaining life (linear RUL, last 200 cycles). Curves that bend near zero RUL are the degradation signature. The shaded band is one standard deviation across engines.

In [242]:
late_life = train_prepared_df[train_prepared_df["RUL_linear"] < 200]
fig = make_subplots(rows=2, cols=3, subplot_titles=ranked_sensors[:6])
for position, sensor in enumerate(ranked_sensors[:6]):
    by_rul = late_life.groupby("RUL_linear")[sensor].agg(["mean", "std"]).fillna(0.0)
    row, col = position // 3 + 1, position % 3 + 1
    fig.add_trace(go.Scatter(x=by_rul.index, y=by_rul["mean"] + by_rul["std"], mode="lines", line=dict(width=0), showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=by_rul.index, y=by_rul["mean"] - by_rul["std"], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(31,119,180,0.2)", showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=by_rul.index, y=by_rul["mean"], mode="lines", line=dict(color="rgb(31,119,180)"), showlegend=False), row=row, col=col)
    fig.update_xaxes(title_text="RUL (linear)", autorange="reversed", row=row, col=col)
fig.update_layout(height=600, width=1000)
fig.show()

## 7.4 Piecewise RUL Analysis
The target is `RUL = min(maximum_cycle_of_engine - current_cycle, RUL_CAP)`.

**Why cap it?** A new engine looks the same at 300 cycles from failure as at 200. A linear target forces the model to predict differences the sensors do not show, and the huge early values dominate the squared error. The plot below shows the average distance of the normalized sensors from their healthy level by remaining life: it is flat for large RUL and rises only near failure, which is where the cap belongs.

The cap is applied to **training and validation labels only**. Official test RUL stays uncapped for the final score, and test engines never get a label built from their own (unknown) future.

In [243]:
distance_from_healthy = train_prepared_df[live_sensors].abs().mean(axis=1)
rul_bin_edges = [-1, 24, 49, 74, 99, 124, 149, 199, 10 ** 6]
rul_bins = pd.cut(train_prepared_df["RUL_linear"], bins=rul_bin_edges)
degradation_curve = distance_from_healthy.groupby(rul_bins, observed=True).mean()
engine_one = train_prepared_df[train_prepared_df["unit"] == train_prepared_df["unit"].min()]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Target of one engine", "Sensors move only near failure"))
fig.add_trace(go.Scatter(x=engine_one["cycle"], y=engine_one["RUL_linear"], mode="lines", name="linear RUL"), row=1, col=1)
fig.add_trace(go.Scatter(x=engine_one["cycle"], y=engine_one["RUL"], mode="lines", name=f"capped at {RUL_CAP}"), row=1, col=1)
fig.add_trace(go.Scatter(x=[str(interval) for interval in degradation_curve.index], y=degradation_curve.values, mode="lines+markers", name="mean |normalized sensor|"), row=1, col=2)
fig.update_xaxes(title_text="cycle", row=1, col=1); fig.update_yaxes(title_text="RUL", row=1, col=1)
fig.update_xaxes(title_text="linear RUL bin", row=1, col=2); fig.update_yaxes(title_text="mean |normalized sensor|", row=1, col=2)
fig.update_layout(height=380, width=1100)
fig.show()

# 08. Feature Engineering for Traditional Machine Learning
Random Forest and XGBoost need a flat table (one row per sample). A single cycle has no memory, so each row summarizes the **past window** of one engine.


## 8.1 Sliding-Window Preparation
One window per cycle, built **for each engine separately**, ending at that cycle. Windows never mix two engines. Training and validation use every cycle; the test set uses only the last window of each engine (the official protocol).

In [244]:
train_windows, train_window_units, train_window_cycles = build_windows(train_prepared_df, live_sensors, WINDOW_SIZE)
print("train windows:", train_windows.shape, "= (windows, window_size, sensors)")

train windows: (43464, 30, 14) = (windows, window_size, sensors)


Validation windows.

In [245]:
val_windows, val_window_units, val_window_cycles = build_windows(val_prepared_df, live_sensors, WINDOW_SIZE)
print("validation windows:", val_windows.shape)

validation windows: (10295, 30, 14)


Test windows (final window of each engine only).

In [246]:
test_windows, test_window_units, test_window_cycles = build_windows(test_prepared_df, live_sensors, WINDOW_SIZE, last_only=True)
print("test windows (last window per engine):", test_windows.shape)

test windows (last window per engine): (259, 30, 14)


## 8.2 Window Length and Short-Sequence Handling
The window length is `WINDOW_SIZE`. An engine with fewer cycles than the window, or the first cycles of any engine, would not have a full window, so the missing earlier steps are **left-padded with the engine's first observation**. Training, validation, and test use the same rule. Padding repeats a value instead of inventing one, and it never touches another engine.

In [247]:
for name, windows in (("train", train_windows), ("validation", val_windows), ("test", test_windows)):
    assert windows.shape[1] == WINDOW_SIZE and windows.shape[2] == len(live_sensors), f"{name}: wrong window shape {windows.shape}"
assert len(train_windows) == len(train_prepared_df) and len(val_windows) == len(val_prepared_df), "one window per cycle expected"
assert len(test_windows) == test_prepared_df["unit"].nunique(), "one test window per engine expected"
first_engine = train_prepared_df[train_prepared_df["unit"] == train_prepared_df["unit"].min()]
first_values = first_engine[live_sensors].to_numpy(dtype=np.float32)
assert np.allclose(train_windows[0], np.repeat(first_values[:1], WINDOW_SIZE, axis=0)), "first window of an engine must be pure padding"
assert np.allclose(train_windows[WINDOW_SIZE - 1], first_values[:WINDOW_SIZE]), "first full window must hold the first real cycles"
short_engines = test_prepared_df.groupby("unit").size().lt(WINDOW_SIZE).sum()
print("test engines shorter than the window (padded):", int(short_engines))

test engines shorter than the window (padded): 6


## 8.3 Window-Level Target Assignment
Each window takes the RUL of its **last** cycle. Because windows are built one per row, the targets are the rows' capped `RUL` values in the same order. The validation "tail" (true RUL below `VALIDATION_TAIL`) mimics the test set, whose engines stop shortly before failure.

In [248]:
y_train = train_prepared_df["RUL"].to_numpy(dtype=np.float32)
y_val = val_prepared_df["RUL"].to_numpy(dtype=np.float32)
val_tail_mask = (val_prepared_df["RUL_linear"] < VALIDATION_TAIL).to_numpy()
y_val_tail = y_val[val_tail_mask]
groups_train = train_prepared_df["unit"].to_numpy()  # engine ids, used for group-aware cross-validation
assert np.array_equal(train_window_units, groups_train) and np.array_equal(train_window_cycles, train_prepared_df["cycle"].to_numpy())
assert np.array_equal(val_window_units, val_prepared_df["unit"].to_numpy()) and np.array_equal(val_window_cycles, val_prepared_df["cycle"].to_numpy())
print(f"y_train {y_train.shape} | y_val {y_val.shape} | validation tail rows {int(val_tail_mask.sum())} of {len(y_val)}")

y_train (43464,) | y_val (10295,) | validation tail rows 6500 of 10295


## 8.4 Traditional ML Feature Extraction
Per sensor and window: last value, mean, standard deviation, minimum, maximum, first-to-last change, and linear-trend slope. Extra features: the `cycle` number (engine age) and, for multi-condition datasets, the `condition` label. All values use only the current and earlier cycles.

In [249]:
extra_names = ["cycle"] + (["condition"] if n_conditions > 1 else [])
train_extra = train_prepared_df[extra_names].to_numpy()
X_train, feature_names = extract_window_features(train_windows, train_extra, extra_names, live_sensors)
print("train feature matrix:", X_train.shape)

train feature matrix: (43464, 100)


Features of the validation windows.

In [250]:
val_extra = val_prepared_df[extra_names].to_numpy()
X_val, val_feature_names = extract_window_features(val_windows, val_extra, extra_names, live_sensors)
print("validation feature matrix:", X_val.shape)

validation feature matrix: (10295, 100)


Features of the test windows (one row per engine).

In [251]:
test_last_rows = test_prepared_df.groupby("unit").tail(1)
test_extra = test_last_rows[extra_names].to_numpy()
X_test, test_feature_names = extract_window_features(test_windows, test_extra, extra_names, live_sensors)
print("test feature matrix (one row per engine):", X_test.shape)

test feature matrix (one row per engine): (259, 100)


## 8.5 Feature Matrix Preparation
DataFrames with column names keep feature order identical everywhere and make importances readable.

In [252]:
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_val_df = pd.DataFrame(X_val, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)
assert feature_names == val_feature_names == test_feature_names, "feature names or order differ between partitions"
print(len(feature_names), "features, e.g.", feature_names[:3], "...", feature_names[-3:])

100 features, e.g. ['T24_last', 'T30_last', 'T50_last'] ... ['W32_slope', 'cycle', 'condition']


## 8.6 Feature Verification
Shape checks, finite values, and a check that no target information entered the predictors.

In [253]:
assert X_train_df.shape == (len(y_train), 7 * len(live_sensors) + len(extra_names)), "unexpected training matrix shape"
assert X_val_df.shape[1] == X_train_df.shape[1] == X_test_df.shape[1], "feature count differs between partitions"
assert X_test_df.shape[0] == test_prepared_df["unit"].nunique(), "one test row per engine expected"
assert all(np.isfinite(frame.to_numpy()).all() for frame in (X_train_df, X_val_df, X_test_df)), "non-finite feature values"
assert not any("RUL" in name for name in feature_names), "a target column leaked into the predictors"
assert X_train_df.dtypes.nunique() == 1 and str(X_train_df.dtypes.iloc[0]) == "float32", "unexpected feature dtype"
print("feature checks passed")
X_train_df.iloc[:3, :6]

feature checks passed


,T24_last,T30_last,T50_last,P30_last,Nf_last,Nc_last
0,-1.098345,-1.459887,0.765083,0.424415,-1.465770,-0.869757
1,0.445908,-0.229703,-0.275121,-0.264844,-1.387548,-1.381210
2,1.244847,-1.128019,-0.429356,0.637880,-1.197962,-1.018961


## 8.7 Validation Helper and Results Registry
Every model is scored the same way: on **all** validation rows and on the test-like **tail**. The tail RMSE is the model-selection metric in section 16. `nasa_mean` is the NASA score per prediction (the summed score depends on how many rows are scored).

In [254]:
def evaluate_validation(val_predictions):
    """Score validation predictions on the test-like tail and on all validation rows."""
    val_predictions = np.asarray(val_predictions, dtype=float)
    assert val_predictions.shape == y_val.shape, "validation predictions must match the validation targets"
    return {"tail": evaluate_predictions(y_val_tail, val_predictions[val_tail_mask]),
            "all": evaluate_predictions(y_val, val_predictions)}


def scores_table(named_scores):
    """Small table that compares the validation scores of several named models."""
    rows = {}
    for name, scores in named_scores.items():
        rows[name] = {"tail RMSE": scores["tail"]["rmse"], "tail MAE": scores["tail"]["mae"], "tail NASA/sample": scores["tail"]["nasa_mean"],
                      "all RMSE": scores["all"]["rmse"], "all MAE": scores["all"]["mae"]}
    return pd.DataFrame(rows).T.round(3)


validation_results = {}  # final validation outcome of each model family, filled in sections 9 to 15
print("validation helpers ready")

validation helpers ready


# 9. Random Forest Regression
A bagged ensemble of decision trees on the engineered window features.


## 9.1 Data Preparation
Same training and validation matrices as every tabular model; the engine id is kept as the cross-validation group so no engine is split across folds.

In [255]:
assert X_train_df.shape[0] == len(y_train) == len(groups_train), "training matrix, target, and groups must align"
print("X_train", X_train_df.shape, "| y_train", y_train.shape, "| engines", len(np.unique(groups_train)))
print("X_val  ", X_val_df.shape, "| y_val", y_val.shape)

X_train (43464, 100) | y_train (43464,) | engines 208
X_val   (10295, 100) | y_val (10295,)


## 9.2 Model Initialization
A forest averages many decorrelated trees, handles mixed feature scales without scaling, and gives a strong baseline. `min_samples_leaf` limits overfitting, and `max_features` decorrelates the trees.

In [256]:
rf_baseline = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=RANDOM_STATE)
print(rf_baseline)

RandomForestRegressor(max_features=0.5, min_samples_leaf=5, n_estimators=150,
                      n_jobs=-1, random_state=42)


## 9.3 Model Training
Fit on the training partition only.

In [257]:
rf_baseline.fit(X_train_df, y_train)
print("Random Forest baseline trained on", len(X_train_df), "rows")

Random Forest baseline trained on 43464 rows


## 9.4 Making Predictions
Predict the validation rows. Predictions are clipped at 0 because RUL cannot be negative.

In [258]:
rf_baseline_val_pred = np.clip(rf_baseline.predict(X_val_df), 0.0, None)
assert rf_baseline_val_pred.shape == y_val.shape and np.isfinite(rf_baseline_val_pred).all()
print("predictions:", rf_baseline_val_pred.shape, "| range", round(float(rf_baseline_val_pred.min()), 1), "to", round(float(rf_baseline_val_pred.max()), 1))

predictions: (10295,) | range 0.8 to 125.0


## 9.5 Validation Evaluation
Tail metrics are the ones that count: they use the same RUL range as the test set. Errors on the capped plateau are easy and would flatter the model, so they are reported separately in the `all` columns.

In [259]:
rf_baseline_scores = evaluate_validation(rf_baseline_val_pred)
scores_table({"Random Forest baseline": rf_baseline_scores})

,tail RMSE,tail MAE,tail NASA/sample,all RMSE,all MAE
Random Forest baseline,16.623,12.399,7.054,14.578,10.094


## 9.6 Feature Importance Analysis
Impurity-based importance: how much each feature reduced the squared error across all trees. It is a guide, not a causal statement, and it can favor features with many distinct values.

In [260]:
rf_importance = pd.Series(rf_baseline.feature_importances_, index=feature_names).sort_values(ascending=False)
rf_top_features = rf_importance.head(15).iloc[::-1]
fig = go.Figure(go.Bar(x=rf_top_features.values, y=rf_top_features.index, orientation="h"))
fig.update_layout(height=450, width=800, xaxis_title="importance", title_text="Random Forest: top 15 features")
fig.show()

## 9.7 Hyperparameter Tuning
Randomized search with `GroupKFold` on the **training partition** (folds split by engine). The validation partition is not used inside the search. Search size is controlled by the configuration cell.

In [ ]:
rf_grid_size = int(np.prod([len(values) for values in RF_GRID.values()]))
rf_search = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE), param_distributions=RF_GRID,
    n_iter=min(RF_SEARCH_ITER, rf_grid_size), cv=GroupKFold(n_splits=CV_FOLDS),
    scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=1, refit=True)
rf_search.fit(X_train_df, y_train, groups=groups_train)  # groups = engine ids
rf_cv_table = pd.DataFrame(rf_search.cv_results_)[["params", "mean_test_score", "std_test_score"]].assign(cv_rmse=lambda t: -t["mean_test_score"])
print("best params:", rf_search.best_params_, "| CV RMSE", round(-rf_search.best_score_, 3))
rf_cv_table.sort_values("cv_rmse").head(5)[["params", "cv_rmse"]]

## 9.8 Best Model Selection
The baseline and the tuned model (refit on the whole training partition by the search) are compared on the validation tail. The better one is kept.

In [ ]:
rf_tuned_val_pred = np.clip(rf_search.best_estimator_.predict(X_val_df), 0.0, None)
rf_tuned_scores = evaluate_validation(rf_tuned_val_pred)
if rf_tuned_scores["tail"]["rmse"] <= rf_baseline_scores["tail"]["rmse"]:
    rf_best_model, rf_best_scores, rf_best_val_pred, rf_best_label = rf_search.best_estimator_, rf_tuned_scores, rf_tuned_val_pred, "tuned"
else:
    rf_best_model, rf_best_scores, rf_best_val_pred, rf_best_label = rf_baseline, rf_baseline_scores, rf_baseline_val_pred, "baseline"
rf_best_params = rf_best_model.get_params()
validation_results["Random Forest"] = {"tail": rf_best_scores["tail"], "all": rf_best_scores["all"], "family": "random_forest",
                                "params": {k: rf_best_params[k] for k in rf_search.best_params_}, "variant": rf_best_label}
display(scores_table({"Random Forest baseline": rf_baseline_scores, "Random Forest tuned": rf_tuned_scores}))
print("selected:", rf_best_label, "| params:", validation_results["Random Forest"]["params"])

## 9.9 Saving Model Artifacts
The model is saved together with the feature names and order, the window size, the RUL cap, and the preprocessing components.

In [ ]:
rf_path = ARTIFACT_PATH / f"random_forest_{DATASET_ID.lower()}.joblib"
joblib.dump({"model": rf_best_model, "feature_names": feature_names, "window_size": WINDOW_SIZE, "rul_cap": RUL_CAP,
             "model_config": validation_results["Random Forest"]["params"], "preprocessor": preprocessor}, rf_path)
print("saved", rf_path.name, f"({rf_path.stat().st_size / 1024:.0f} KB)")

## 9.10 Loading the Saved Model
The model is read back from disk and its feature list is checked against the one used for training.

In [ ]:
# rf_artifact = joblib.load(rf_path)
# rf_loaded = rf_artifact["model"]
# assert rf_artifact["feature_names"] == feature_names, "feature names or order changed"
# assert rf_artifact["window_size"] == WINDOW_SIZE and rf_artifact["rul_cap"] == RUL_CAP
# print("loaded", type(rf_loaded).__name__, "with", len(rf_artifact["feature_names"]), "features")

## 9.11 Inference Using the Loaded Model
The loaded model must reproduce the in-memory validation predictions.

In [ ]:
# rf_loaded_val_pred = np.clip(rf_loaded.predict(X_val_df[rf_artifact["feature_names"]]), 0.0, None)
# assert np.allclose(rf_loaded_val_pred, rf_best_val_pred, atol=1e-4), "loaded Random Forest model does not reproduce its predictions"
# print("loaded model reproduces", len(rf_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(rf_loaded_val_pred)["tail"]["rmse"], 3))

# 10. XGBoost Regression
A gradient-boosted tree ensemble on the same engineered window features.


## 10.1 Data Preparation
Same training and validation matrices as every tabular model; the engine id is kept as the cross-validation group so no engine is split across folds.

In [ ]:
assert X_train_df.shape[0] == len(y_train) == len(groups_train), "training matrix, target, and groups must align"
print("X_train", X_train_df.shape, "| y_train", y_train.shape, "| engines", len(np.unique(groups_train)))
print("X_val  ", X_val_df.shape, "| y_val", y_val.shape)

## 10.2 Model Initialization
Gradient boosting builds trees one after another, each correcting the previous errors, and usually beats a forest when tuned. `learning_rate` and `n_estimators` trade off against each other; `subsample` and `colsample_bytree` add randomness against overfitting. The CPU `hist` method keeps training fast.

In [ ]:
xgb_baseline = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                                objective="reg:squarederror", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE)
print(xgb_baseline.get_params()["n_estimators"], "trees | depth", xgb_baseline.get_params()["max_depth"], "| learning rate", xgb_baseline.get_params()["learning_rate"])

## 10.3 Model Training
Fit on the training partition only.

In [ ]:
xgb_baseline.fit(X_train_df, y_train)
print("XGBoost baseline trained on", len(X_train_df), "rows")

## 10.4 Making Predictions
Predict the validation rows. Predictions are clipped at 0 because RUL cannot be negative.

In [ ]:
xgb_baseline_val_pred = np.clip(xgb_baseline.predict(X_val_df), 0.0, None)
assert xgb_baseline_val_pred.shape == y_val.shape and np.isfinite(xgb_baseline_val_pred).all()
print("predictions:", xgb_baseline_val_pred.shape, "| range", round(float(xgb_baseline_val_pred.min()), 1), "to", round(float(xgb_baseline_val_pred.max()), 1))

## 10.5 Validation Evaluation
Tail metrics are the ones that count: they use the same RUL range as the test set. Errors on the capped plateau are easy and would flatter the model, so they are reported separately in the `all` columns.

In [ ]:
xgb_baseline_scores = evaluate_validation(xgb_baseline_val_pred)
scores_table({"XGBoost baseline": xgb_baseline_scores})

## 10.6 Feature Importance Analysis
Gain-based importance: the average loss reduction when a feature is used in a split. Compare it with the forest's ranking; features that rank high in both are the most trustworthy.

In [ ]:
xgb_importance = pd.Series(xgb_baseline.feature_importances_, index=feature_names).sort_values(ascending=False)
xgb_top_features = xgb_importance.head(15).iloc[::-1]
fig = go.Figure(go.Bar(x=xgb_top_features.values, y=xgb_top_features.index, orientation="h"))
fig.update_layout(height=450, width=800, xaxis_title="importance", title_text="XGBoost: top 15 features")
fig.show()

## 10.7 Hyperparameter Tuning
Randomized search with `GroupKFold` on the **training partition** (folds split by engine). The validation partition is not used inside the search. Search size is controlled by the configuration cell.

In [ ]:
xgb_grid_size = int(np.prod([len(values) for values in XGB_GRID.values()]))
xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    param_distributions=XGB_GRID, n_iter=min(XGB_SEARCH_ITER, xgb_grid_size), cv=GroupKFold(n_splits=CV_FOLDS),
    scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=1, refit=True)
xgb_search.fit(X_train_df, y_train, groups=groups_train)  # groups = engine ids
xgb_cv_table = pd.DataFrame(xgb_search.cv_results_)[["params", "mean_test_score"]].assign(cv_rmse=lambda t: -t["mean_test_score"])
print("best params:", xgb_search.best_params_, "| CV RMSE", round(-xgb_search.best_score_, 3))
xgb_cv_table.sort_values("cv_rmse").head(5)[["params", "cv_rmse"]]

## 10.8 Best Model Selection
The baseline and the tuned model (refit on the whole training partition by the search) are compared on the validation tail. The better one is kept.

In [ ]:
xgb_tuned_val_pred = np.clip(xgb_search.best_estimator_.predict(X_val_df), 0.0, None)
xgb_tuned_scores = evaluate_validation(xgb_tuned_val_pred)
if xgb_tuned_scores["tail"]["rmse"] <= xgb_baseline_scores["tail"]["rmse"]:
    xgb_best_model, xgb_best_scores, xgb_best_val_pred, xgb_best_label = xgb_search.best_estimator_, xgb_tuned_scores, xgb_tuned_val_pred, "tuned"
else:
    xgb_best_model, xgb_best_scores, xgb_best_val_pred, xgb_best_label = xgb_baseline, xgb_baseline_scores, xgb_baseline_val_pred, "baseline"
xgb_best_params = xgb_best_model.get_params()
validation_results["XGBoost"] = {"tail": xgb_best_scores["tail"], "all": xgb_best_scores["all"], "family": "xgboost",
                                "params": {k: xgb_best_params[k] for k in xgb_search.best_params_}, "variant": xgb_best_label}
display(scores_table({"XGBoost baseline": xgb_baseline_scores, "XGBoost tuned": xgb_tuned_scores}))
print("selected:", xgb_best_label, "| params:", validation_results["XGBoost"]["params"])

## 10.9 Saving Model Artifacts
The model is saved together with the feature names and order, the window size, the RUL cap, and the preprocessing components.

In [ ]:
xgb_path = ARTIFACT_PATH / f"xgboost_{DATASET_ID.lower()}.json"
xgb_meta_path = ARTIFACT_PATH / f"xgboost_{DATASET_ID.lower()}_meta.joblib"
xgb_best_model.save_model(xgb_path)  # XGBoost's own JSON format is portable across versions
joblib.dump({"feature_names": feature_names, "window_size": WINDOW_SIZE, "rul_cap": RUL_CAP,
             "model_config": validation_results["XGBoost"]["params"], "preprocessor": preprocessor}, xgb_meta_path)
print("saved", xgb_path.name, "and", xgb_meta_path.name)

## 10.10 Loading the Saved Model
The model is read back from disk and its feature list is checked against the one used for training.

In [ ]:
# xgb_artifact = joblib.load(xgb_meta_path)
# xgb_loaded = xgb.XGBRegressor()
# xgb_loaded.load_model(xgb_path)
# assert xgb_artifact["feature_names"] == feature_names, "feature names or order changed"
# assert xgb_loaded.n_features_in_ == len(feature_names), "loaded model expects a different number of features"
# print("loaded XGBRegressor with", xgb_loaded.n_features_in_, "features")

## 10.11 Inference Using the Loaded Model
The loaded model must reproduce the in-memory validation predictions.

In [ ]:
# xgb_loaded_val_pred = np.clip(xgb_loaded.predict(X_val_df[xgb_artifact["feature_names"]]), 0.0, None)
# assert np.allclose(xgb_loaded_val_pred, xgb_best_val_pred, atol=1e-4), "loaded XGBoost model does not reproduce its predictions"
# print("loaded model reproduces", len(xgb_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(xgb_loaded_val_pred)["tail"]["rmse"], 3))

# 11. Feature Engineering for Deep Learning
Neural networks read the **raw normalized window** as a time series, not the summary table of section 8. The windows built in 8.1 are already per-engine, left-padded, and aligned with the targets, so they are reused here and converted to tensors.


## 11.1 Sequence Preparation
Sequences are chronological: each engine's rows are sorted by cycle (verified in 3.3), and each window runs from the oldest to the newest cycle.

In [ ]:
sequence_features = live_sensors
print("sequence features:", len(sequence_features), "| window length:", WINDOW_SIZE)
assert train_windows.dtype == np.float32 and val_windows.dtype == np.float32 and test_windows.dtype == np.float32, "windows must be float32"

## 11.2 Sliding-Window Generation
Windows were generated per engine in 8.1 (`build_windows`). The check below confirms that the last time step of a window is the row it belongs to.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
for row_index in rng.integers(0, len(train_prepared_df), size=20):
    assert np.allclose(train_windows[row_index, -1, :], train_prepared_df.loc[row_index, live_sensors].to_numpy(dtype=np.float32)), "window end does not match its row"
print("windows are aligned with their rows:", train_windows.shape, val_windows.shape, test_windows.shape)

## 11.3 RUL Target Alignment
The target of a window is the RUL at its **final** time step (`y_train`, `y_val` from 8.3).

In [ ]:
assert len(train_windows) == len(y_train) and len(val_windows) == len(y_val), "windows and targets must have the same length"
print("targets aligned | y_train", y_train.shape, "| y_val", y_val.shape, "| y_val_tail", y_val_tail.shape)

## 11.4 Padding and Truncation
Windows are never truncated, and they all have the same length. Early cycles are left-padded with the first observation (8.2). A model therefore sees the same tensor shape for every sample, and padding is applied the same way in training, validation, and test.

In [ ]:
assert {train_windows.shape[1], val_windows.shape[1], test_windows.shape[1]} == {WINDOW_SIZE}, "all partitions must use the same window length"
padded_fraction = float((train_prepared_df["cycle"] < WINDOW_SIZE).mean())
print(f"fraction of training windows that contain padding: {padded_fraction:.1%}")

## 11.5 Tensor Conversion
`float32` tensors. `torch.from_numpy` shares memory with the arrays, which avoids copying large window tensors.

In [ ]:
x_train_t = torch.from_numpy(train_windows)
y_train_t = torch.from_numpy(y_train.copy())  # copy: pandas arrays can be read-only
x_val_t = torch.from_numpy(val_windows)
y_val_t = torch.from_numpy(y_val.copy())
x_val_tail_t = x_val_t[torch.from_numpy(val_tail_mask)]
x_test_t = torch.from_numpy(test_windows)
print("x_train_t", tuple(x_train_t.shape), x_train_t.dtype, "| y_train_t", tuple(y_train_t.shape))

## 11.6 Dataset and DataLoader Preparation
`make_train_loader` creates a shuffled loader with its **own seeded generator**, so every model (and every tuning trial) sees the same batch order and results are repeatable. Validation and test loaders are not shuffled.

In [ ]:
def make_train_loader(x_tensor, y_tensor):
    """Shuffled training loader with a fresh, seeded random generator."""
    return DataLoader(TensorDataset(x_tensor, y_tensor), batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(RANDOM_STATE))


train_loader = make_train_loader(x_train_t, y_train_t)
val_loader = DataLoader(TensorDataset(x_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(x_test_t, torch.zeros(len(x_test_t))), batch_size=BATCH_SIZE, shuffle=False)  # no test labels are used here
print("batches per epoch (train):", len(train_loader), "| validation:", len(val_loader), "| test:", len(test_loader))

## 11.7 Tensor Shape Verification
Meaning of each dimension:

| Tensor | Shape | Meaning |
|---|---|---|
| inputs `x` | `(samples, window_size, features)` | samples = windows, window_size = cycles in the window, features = live sensors |
| targets `y` | `(samples,)` | RUL at the last cycle of each window |
| model output | `(batch,)` | one RUL / RUL_CAP value per window |

In [ ]:
n_features = len(live_sensors)
assert x_train_t.shape == (len(y_train), WINDOW_SIZE, n_features) and y_train_t.shape == (len(y_train),)
assert x_val_t.shape == (len(y_val), WINDOW_SIZE, n_features) and y_val_t.shape == (len(y_val),)
assert x_test_t.shape == (test_prepared_df["unit"].nunique(), WINDOW_SIZE, n_features)
assert x_val_tail_t.shape[0] == len(y_val_tail)
assert all(t.dtype == torch.float32 for t in (x_train_t, y_train_t, x_val_t, y_val_t, x_test_t))
assert all(torch.isfinite(t).all() for t in (x_train_t, y_train_t, x_val_t, x_test_t)), "non-finite tensor values"
batch_x, batch_y = next(iter(train_loader))
assert batch_x.shape[1:] == (WINDOW_SIZE, n_features) and batch_x.shape[0] == batch_y.shape[0] <= BATCH_SIZE
print("one batch:", tuple(batch_x.shape), tuple(batch_y.shape))

## 11.8 Model Factory and Training Utilities
One factory builds every network from its name and configuration, so training, tuning, saving, and loading always construct the same architecture. (The network classes themselves are defined in sections 12 to 15 before the factory is first called.)

In [ ]:
def count_parameters(model):
    """Number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_model(model_name, input_dim, config):
    """Create one of the four sequence networks from its name and configuration dictionary."""
    if model_name == "LSTM":
        return LSTMRegressor(input_dim, **config)
    if model_name == "CNN-LSTM":
        return CNNLSTMRegressor(input_dim, **config)
    if model_name == "TCN":
        return TCNRegressor(input_dim, **config)
    if model_name == "Transformer":
        return TransformerRegressor(input_dim, max_len=WINDOW_SIZE, **config)
    raise ValueError(f"unknown model name: {model_name}")


print("factory ready for:", list(DL_CONFIGS))

# 12. LSTM Regression
An LSTM reads the window one cycle at a time and keeps a memory of earlier cycles through gates that decide what to remember and what to forget. That suits degradation, which builds up slowly over many cycles.

## 12.1 Data Preparation
Same tensors and loader settings as every sequence model. The check confirms the model input will be `(batch, window_size, features)`.

In [ ]:
lstm_batch_x, lstm_batch_y = next(iter(make_train_loader(x_train_t, y_train_t)))
assert lstm_batch_x.shape[1:] == (WINDOW_SIZE, n_features), "unexpected batch shape"
print("input batch:", tuple(lstm_batch_x.shape), "| targets:", tuple(lstm_batch_y.shape))

## 12.2 Network Architecture
Data flow: `(batch, window, features)` -> stacked LSTM -> hidden state of the **last** time step `(batch, hidden)` -> dense head -> `(batch,)`. `hidden` is the memory size, `layers` the depth, and `dropout` the regularization between layers.

In [ ]:
class LSTMRegressor(nn.Module):
    """Stacked LSTM; the hidden state of the last cycle feeds a small dense head."""

    def __init__(self, input_dim, hidden, layers, dropout):
        super().__init__()
        # inter-layer dropout only exists when there is more than one LSTM layer
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        sequence_out, _ = self.lstm(x)                           # (batch, window, hidden)
        return self.head(sequence_out[:, -1, :]).squeeze(-1)    # last cycle only -> (batch,)

## 12.3 Model Initialization
Create the network with the **default** configuration (first entry in `DL_CONFIGS`) and push one batch through it to confirm the output shape `(batch,)`.

In [ ]:
set_seed(RANDOM_STATE)
lstm_default_config = DL_CONFIGS["LSTM"][0]
lstm_model = build_model("LSTM", n_features, lstm_default_config)
with torch.no_grad():
    lstm_sample_output = lstm_model.eval()(x_train_t[:8])
assert lstm_sample_output.shape == (8,), f"expected output shape (8,), got {tuple(lstm_sample_output.shape)}"
print(lstm_model)
print("trainable parameters:", count_parameters(lstm_model), "| config:", lstm_default_config)

## 12.4 Loss Function and Optimizer
Mean squared error on targets scaled to `RUL / RUL_CAP`: it matches the RMSE metric and keeps the loss near 0 to 1. Adam adapts the step size per weight, and a small weight decay discourages overfitting. Gradients are clipped inside `train_network`.

In [ ]:
lstm_loss = nn.MSELoss()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
print("loss: MSE | optimizer: Adam | learning rate:", LEARNING_RATE)

## 12.5 Training Loop
Each epoch: forward pass, loss, backward pass, gradient clipping, optimizer step. After every epoch the model is scored on the validation tail (the epoch-level validation of the next step).

In [ ]:
lstm_loader = make_train_loader(x_train_t, y_train_t)
lstm_history = train_network(lstm_model, lstm_optimizer, lstm_loss, lstm_loader, EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail)
print("epochs run:", len(lstm_history["epoch"]), "| best validation-tail RMSE:", round(lstm_history["best_rmse"], 3))

## 12.6 Epoch-Level Validation
Training loss should fall steadily. Validation RMSE that stops improving while the training loss keeps falling means overfitting.

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=lstm_history["epoch"], y=lstm_history["train_loss"], mode="lines+markers", name="train loss (scaled MSE)"), secondary_y=False)
fig.add_trace(go.Scatter(x=lstm_history["epoch"], y=lstm_history["val_rmse"], mode="lines+markers", name="validation-tail RMSE (cycles)"), secondary_y=True)
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="train loss (scaled MSE)", secondary_y=False)
fig.update_yaxes(title_text="validation-tail RMSE (cycles)", secondary_y=True)
fig.update_layout(height=380, width=800, title_text="LSTM: learning curves")
fig.show()

## 12.7 Early Stopping
Training stops when the validation-tail RMSE has not improved for `EARLY_STOPPING_PATIENCE` epochs, and the weights of the best epoch are restored. This is the cheapest protection against overfitting.

In [ ]:
lstm_stopped_early = len(lstm_history["epoch"]) < EPOCHS
print("stopped early:", lstm_stopped_early, "| epochs run:", len(lstm_history["epoch"]), "| best epoch:", lstm_history["best_epoch"])
assert 1 <= lstm_history["best_epoch"] <= len(lstm_history["epoch"]), "no valid best epoch was recorded"

## 12.8 Prediction and Inference
Inference uses `model.eval()` (dropout and batch norm in inference mode) and `torch.no_grad()` (no gradient bookkeeping). The output is rescaled to cycles.

In [ ]:
lstm_val_pred = predict_network(lstm_model, x_val_t)
assert lstm_val_pred.shape == y_val.shape and np.isfinite(lstm_val_pred).all(), "invalid validation predictions"
print("validation predictions:", lstm_val_pred.shape, "| range", round(float(lstm_val_pred.min()), 1), "to", round(float(lstm_val_pred.max()), 1))

## 12.9 Model Evaluation
Validation metrics on the test-like tail and on all rows, plus predicted versus true RUL for the tail.

In [ ]:
lstm_scores = evaluate_validation(lstm_val_pred)
display(scores_table({"LSTM (default config)": lstm_scores}))
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_val_tail, y=lstm_val_pred[val_tail_mask], mode="markers", marker=dict(size=4, opacity=0.4), name="validation tail"))
fig.add_trace(go.Scatter(x=[0, RUL_CAP], y=[0, RUL_CAP], mode="lines", line=dict(dash="dash", color="black"), name="ideal"))
fig.update_layout(height=450, width=520, xaxis_title="true RUL", yaxis_title="predicted RUL", title_text="LSTM: validation tail")
fig.show()

## 12.10 Hyperparameter Tuning
A small, fixed list of configurations (`DL_CONFIGS`) is trained for `TUNE_EPOCHS` epochs each, with the same seed, the same batches, and the **same validation split**. The best configuration is the one with the lowest validation-tail RMSE.

In [ ]:
lstm_trials = []
for lstm_trial_config in DL_CONFIGS["LSTM"]:
    set_seed(RANDOM_STATE)
    lstm_trial_model = build_model("LSTM", n_features, lstm_trial_config)
    lstm_trial_optimizer = torch.optim.Adam(lstm_trial_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    lstm_trial_history = train_network(lstm_trial_model, lstm_trial_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                       TUNE_EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    lstm_trials.append({"config": lstm_trial_config, "val_tail_rmse": lstm_trial_history["best_rmse"],
                       "best_epoch": lstm_trial_history["best_epoch"], "parameters": count_parameters(lstm_trial_model)})
lstm_trials_table = pd.DataFrame(lstm_trials).sort_values("val_tail_rmse")
lstm_trials_table.round(3)

## 12.11 Best Configuration Selection
If the best trial is not the default, that configuration is retrained for the full `EPOCHS` with early stopping. It replaces the default model only if its validation-tail RMSE is lower.

In [ ]:
lstm_best_config = lstm_trials_table.iloc[0]["config"]
lstm_final_model, lstm_final_config, lstm_final_history = lstm_model, lstm_default_config, lstm_history
if lstm_best_config != lstm_default_config:
    set_seed(RANDOM_STATE)
    lstm_candidate = build_model("LSTM", n_features, lstm_best_config)
    lstm_candidate_optimizer = torch.optim.Adam(lstm_candidate.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    lstm_candidate_history = train_network(lstm_candidate, lstm_candidate_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                           EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    if lstm_candidate_history["best_rmse"] < lstm_history["best_rmse"]:
        lstm_final_model, lstm_final_config, lstm_final_history = lstm_candidate, lstm_best_config, lstm_candidate_history
print("selected config:", lstm_final_config, "| validation-tail RMSE:", round(lstm_final_history["best_rmse"], 3), "| best epoch:", lstm_final_history["best_epoch"])

## 12.12 Validation Result of the Selected Model
The selected network is scored once more and stored for the comparison in section 16.

In [ ]:
lstm_final_val_pred = predict_network(lstm_final_model, x_val_t)
lstm_final_scores = evaluate_validation(lstm_final_val_pred)
validation_results["LSTM"] = {"tail": lstm_final_scores["tail"], "all": lstm_final_scores["all"], "family": "nn",
                            "params": lstm_final_config, "best_epoch": lstm_final_history["best_epoch"],
                            "parameters": count_parameters(lstm_final_model)}
scores_table({"LSTM (selected)": lstm_final_scores})

## 12.13 Saving Model Artifacts
Weights (as plain tensors), the architecture configuration, the input size, the feature names, the window size, the RUL cap, and the name of the preprocessing file go into one checkpoint. Together with `preprocessor_*.joblib` this is everything needed to reproduce predictions.

In [ ]:
lstm_st_path = ARTIFACT_PATH / f"lstm_{DATASET_ID.lower()}.safetensors"
save_file({key: value.detach().cpu().contiguous() for key, value in lstm_final_model.state_dict().items()},
          str(lstm_st_path),
          metadata={"model_name": "LSTM", "config": json.dumps(lstm_final_config), "input_dim": str(n_features),
                    "feature_names": json.dumps(live_sensors), "window_size": str(WINDOW_SIZE), "rul_cap": str(RUL_CAP),
                    "preprocessor_file": preprocessor_path.name})
print("saved", lstm_st_path.name, f"({lstm_st_path.stat().st_size / 1024:.0f} KB)")

## 12.14 Loading the Saved Model
The architecture is rebuilt from the stored configuration and **checked against the stored weights** (same parameter names and shapes) before the weights are loaded.

In [ ]:
# lstm_checkpoint = torch.load(lstm_path, map_location="cpu", weights_only=True)
# assert lstm_checkpoint["model_name"] == "LSTM" and lstm_checkpoint["window_size"] == WINDOW_SIZE and lstm_checkpoint["rul_cap"] == RUL_CAP
# lstm_loaded = build_model("LSTM", lstm_checkpoint["input_dim"], lstm_checkpoint["config"])
# lstm_expected_state = lstm_loaded.state_dict()
# assert list(lstm_expected_state.keys()) == list(lstm_checkpoint["state_dict"].keys()), "parameter names differ from the checkpoint"
# for lstm_key in lstm_expected_state:
#     assert lstm_expected_state[lstm_key].shape == lstm_checkpoint["state_dict"][lstm_key].shape, f"shape mismatch for {lstm_key}"
# lstm_loaded.load_state_dict(lstm_checkpoint["state_dict"])
# lstm_loaded.to(DEVICE).eval()
# print("loaded", lstm_checkpoint["model_name"], "| parameters:", count_parameters(lstm_loaded))

## 12.15 Inference Using the Loaded Model
The reloaded network must reproduce the in-memory validation predictions.

In [ ]:
# lstm_loaded_val_pred = predict_network(lstm_loaded, x_val_t)
# assert np.allclose(lstm_loaded_val_pred, lstm_final_val_pred, atol=1e-3), "loaded LSTM model does not reproduce its predictions"
# print("loaded model reproduces", len(lstm_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(lstm_loaded_val_pred)["tail"]["rmse"], 3))

# 13. Hybrid CNN-LSTM Regression
A 1D convolution first extracts **local patterns across neighboring cycles** (short slopes, sudden jumps). The LSTM then models how those patterns evolve over the whole window.

## 13.1 Data Preparation
Same tensors and loader settings as every sequence model. The check confirms the model input will be `(batch, window_size, features)`.

In [ ]:
cnn_lstm_batch_x, cnn_lstm_batch_y = next(iter(make_train_loader(x_train_t, y_train_t)))
assert cnn_lstm_batch_x.shape[1:] == (WINDOW_SIZE, n_features), "unexpected batch shape"
print("input batch:", tuple(cnn_lstm_batch_x.shape), "| targets:", tuple(cnn_lstm_batch_y.shape))

## 13.2 Network Architecture
Dimension changes matter here. `Conv1d` expects `(batch, channels, time)` but the data is `(batch, time, features)`, so the tensor is transposed before the convolution and transposed back before the LSTM (`batch_first=True`):

`(B, T, F)` -> transpose -> `(B, F, T)` -> Conv1d -> `(B, filters, T)` -> BatchNorm + ReLU -> transpose -> `(B, T, filters)` -> LSTM -> last step `(B, hidden)` -> head -> `(B,)`.

An odd `kernel` with `padding = kernel // 2` keeps the window length. The convolution only looks at cycles inside the window, which all lie at or before the predicted cycle.

In [ ]:
class CNNLSTMRegressor(nn.Module):
    """Conv1d feature extractor followed by a stacked LSTM and a dense head."""

    def __init__(self, input_dim, filters, kernel, hidden, layers, dropout):
        super().__init__()
        assert kernel % 2 == 1, "use an odd kernel so the window length is preserved"
        self.conv = nn.Conv1d(input_dim, filters, kernel, padding=kernel // 2)
        self.norm = nn.BatchNorm1d(filters)
        self.lstm = nn.LSTM(filters, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        z = x.transpose(1, 2)                        # (batch, features, window): Conv1d wants channels first
        z = F.relu(self.norm(self.conv(z)))          # (batch, filters, window)
        z = z.transpose(1, 2)                        # (batch, window, filters): LSTM wants time second
        sequence_out, _ = self.lstm(z)               # (batch, window, hidden)
        return self.head(sequence_out[:, -1, :]).squeeze(-1)

## 13.3 Model Initialization
Create the network with the **default** configuration (first entry in `DL_CONFIGS`) and push one batch through it to confirm the output shape `(batch,)`.

In [ ]:
set_seed(RANDOM_STATE)
cnn_lstm_default_config = DL_CONFIGS["CNN-LSTM"][0]
cnn_lstm_model = build_model("CNN-LSTM", n_features, cnn_lstm_default_config)
with torch.no_grad():
    cnn_lstm_sample_output = cnn_lstm_model.eval()(x_train_t[:8])
assert cnn_lstm_sample_output.shape == (8,), f"expected output shape (8,), got {tuple(cnn_lstm_sample_output.shape)}"
print(cnn_lstm_model)
print("trainable parameters:", count_parameters(cnn_lstm_model), "| config:", cnn_lstm_default_config)

## 13.4 Loss Function and Optimizer
Mean squared error on targets scaled to `RUL / RUL_CAP`: it matches the RMSE metric and keeps the loss near 0 to 1. Adam adapts the step size per weight, and a small weight decay discourages overfitting. Gradients are clipped inside `train_network`.

In [ ]:
cnn_lstm_loss = nn.MSELoss()
cnn_lstm_optimizer = torch.optim.Adam(cnn_lstm_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
print("loss: MSE | optimizer: Adam | learning rate:", LEARNING_RATE)

## 13.5 Training Loop
Each epoch: forward pass, loss, backward pass, gradient clipping, optimizer step. After every epoch the model is scored on the validation tail (the epoch-level validation of the next step).

In [ ]:
cnn_lstm_loader = make_train_loader(x_train_t, y_train_t)
cnn_lstm_history = train_network(cnn_lstm_model, cnn_lstm_optimizer, cnn_lstm_loss, cnn_lstm_loader, EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail)
print("epochs run:", len(cnn_lstm_history["epoch"]), "| best validation-tail RMSE:", round(cnn_lstm_history["best_rmse"], 3))

## 13.6 Epoch-Level Validation
Training loss should fall steadily. Validation RMSE that stops improving while the training loss keeps falling means overfitting.

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=cnn_lstm_history["epoch"], y=cnn_lstm_history["train_loss"], mode="lines+markers", name="train loss (scaled MSE)"), secondary_y=False)
fig.add_trace(go.Scatter(x=cnn_lstm_history["epoch"], y=cnn_lstm_history["val_rmse"], mode="lines+markers", name="validation-tail RMSE (cycles)"), secondary_y=True)
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="train loss (scaled MSE)", secondary_y=False)
fig.update_yaxes(title_text="validation-tail RMSE (cycles)", secondary_y=True)
fig.update_layout(height=380, width=800, title_text="CNN-LSTM: learning curves")
fig.show()

## 13.7 Early Stopping
Training stops when the validation-tail RMSE has not improved for `EARLY_STOPPING_PATIENCE` epochs, and the weights of the best epoch are restored. This is the cheapest protection against overfitting.

In [ ]:
cnn_lstm_stopped_early = len(cnn_lstm_history["epoch"]) < EPOCHS
print("stopped early:", cnn_lstm_stopped_early, "| epochs run:", len(cnn_lstm_history["epoch"]), "| best epoch:", cnn_lstm_history["best_epoch"])
assert 1 <= cnn_lstm_history["best_epoch"] <= len(cnn_lstm_history["epoch"]), "no valid best epoch was recorded"

## 13.8 Prediction and Inference
Inference uses `model.eval()` (dropout and batch norm in inference mode) and `torch.no_grad()` (no gradient bookkeeping). The output is rescaled to cycles.

In [ ]:
cnn_lstm_val_pred = predict_network(cnn_lstm_model, x_val_t)
assert cnn_lstm_val_pred.shape == y_val.shape and np.isfinite(cnn_lstm_val_pred).all(), "invalid validation predictions"
print("validation predictions:", cnn_lstm_val_pred.shape, "| range", round(float(cnn_lstm_val_pred.min()), 1), "to", round(float(cnn_lstm_val_pred.max()), 1))

## 13.9 Model Evaluation
Validation metrics on the test-like tail and on all rows, plus predicted versus true RUL for the tail.

In [ ]:
cnn_lstm_scores = evaluate_validation(cnn_lstm_val_pred)
display(scores_table({"CNN-LSTM (default config)": cnn_lstm_scores}))
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_val_tail, y=cnn_lstm_val_pred[val_tail_mask], mode="markers", marker=dict(size=4, opacity=0.4), name="validation tail"))
fig.add_trace(go.Scatter(x=[0, RUL_CAP], y=[0, RUL_CAP], mode="lines", line=dict(dash="dash", color="black"), name="ideal"))
fig.update_layout(height=450, width=520, xaxis_title="true RUL", yaxis_title="predicted RUL", title_text="CNN-LSTM: validation tail")
fig.show()

## 13.10 Hyperparameter Tuning
A small, fixed list of configurations (`DL_CONFIGS`) is trained for `TUNE_EPOCHS` epochs each, with the same seed, the same batches, and the **same validation split**. The best configuration is the one with the lowest validation-tail RMSE.

In [ ]:
cnn_lstm_trials = []
for cnn_lstm_trial_config in DL_CONFIGS["CNN-LSTM"]:
    set_seed(RANDOM_STATE)
    cnn_lstm_trial_model = build_model("CNN-LSTM", n_features, cnn_lstm_trial_config)
    cnn_lstm_trial_optimizer = torch.optim.Adam(cnn_lstm_trial_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    cnn_lstm_trial_history = train_network(cnn_lstm_trial_model, cnn_lstm_trial_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                       TUNE_EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    cnn_lstm_trials.append({"config": cnn_lstm_trial_config, "val_tail_rmse": cnn_lstm_trial_history["best_rmse"],
                       "best_epoch": cnn_lstm_trial_history["best_epoch"], "parameters": count_parameters(cnn_lstm_trial_model)})
cnn_lstm_trials_table = pd.DataFrame(cnn_lstm_trials).sort_values("val_tail_rmse")
cnn_lstm_trials_table.round(3)

## 13.11 Best Configuration Selection
If the best trial is not the default, that configuration is retrained for the full `EPOCHS` with early stopping. It replaces the default model only if its validation-tail RMSE is lower.

In [ ]:
cnn_lstm_best_config = cnn_lstm_trials_table.iloc[0]["config"]
cnn_lstm_final_model, cnn_lstm_final_config, cnn_lstm_final_history = cnn_lstm_model, cnn_lstm_default_config, cnn_lstm_history
if cnn_lstm_best_config != cnn_lstm_default_config:
    set_seed(RANDOM_STATE)
    cnn_lstm_candidate = build_model("CNN-LSTM", n_features, cnn_lstm_best_config)
    cnn_lstm_candidate_optimizer = torch.optim.Adam(cnn_lstm_candidate.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    cnn_lstm_candidate_history = train_network(cnn_lstm_candidate, cnn_lstm_candidate_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                           EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    if cnn_lstm_candidate_history["best_rmse"] < cnn_lstm_history["best_rmse"]:
        cnn_lstm_final_model, cnn_lstm_final_config, cnn_lstm_final_history = cnn_lstm_candidate, cnn_lstm_best_config, cnn_lstm_candidate_history
print("selected config:", cnn_lstm_final_config, "| validation-tail RMSE:", round(cnn_lstm_final_history["best_rmse"], 3), "| best epoch:", cnn_lstm_final_history["best_epoch"])

## 13.12 Validation Result of the Selected Model
The selected network is scored once more and stored for the comparison in section 16.

In [ ]:
cnn_lstm_final_val_pred = predict_network(cnn_lstm_final_model, x_val_t)
cnn_lstm_final_scores = evaluate_validation(cnn_lstm_final_val_pred)
validation_results["CNN-LSTM"] = {"tail": cnn_lstm_final_scores["tail"], "all": cnn_lstm_final_scores["all"], "family": "nn",
                            "params": cnn_lstm_final_config, "best_epoch": cnn_lstm_final_history["best_epoch"],
                            "parameters": count_parameters(cnn_lstm_final_model)}
scores_table({"CNN-LSTM (selected)": cnn_lstm_final_scores})

## 13.13 Saving Model Artifacts
Weights (as plain tensors), the architecture configuration, the input size, the feature names, the window size, the RUL cap, and the name of the preprocessing file go into one checkpoint. Together with `preprocessor_*.joblib` this is everything needed to reproduce predictions.

In [ ]:
cnn_lstm_st_path = ARTIFACT_PATH / f"cnn_lstm_{DATASET_ID.lower()}.safetensors"
save_file({key: value.detach().cpu().contiguous() for key, value in cnn_lstm_final_model.state_dict().items()},
          str(cnn_lstm_st_path),
          metadata={"model_name": "CNN-LSTM", "config": json.dumps(cnn_lstm_final_config), "input_dim": str(n_features),
                    "feature_names": json.dumps(live_sensors), "window_size": str(WINDOW_SIZE), "rul_cap": str(RUL_CAP),
                    "preprocessor_file": preprocessor_path.name})
print("saved", cnn_lstm_st_path.name, f"({cnn_lstm_st_path.stat().st_size / 1024:.0f} KB)")

## 13.14 Loading the Saved Model
The architecture is rebuilt from the stored configuration and **checked against the stored weights** (same parameter names and shapes) before the weights are loaded.

In [ ]:
# cnn_lstm_checkpoint = torch.load(cnn_lstm_path, map_location="cpu", weights_only=True)
# assert cnn_lstm_checkpoint["model_name"] == "CNN-LSTM" and cnn_lstm_checkpoint["window_size"] == WINDOW_SIZE and cnn_lstm_checkpoint["rul_cap"] == RUL_CAP
# cnn_lstm_loaded = build_model("CNN-LSTM", cnn_lstm_checkpoint["input_dim"], cnn_lstm_checkpoint["config"])
# cnn_lstm_expected_state = cnn_lstm_loaded.state_dict()
# assert list(cnn_lstm_expected_state.keys()) == list(cnn_lstm_checkpoint["state_dict"].keys()), "parameter names differ from the checkpoint"
# for cnn_lstm_key in cnn_lstm_expected_state:
#     assert cnn_lstm_expected_state[cnn_lstm_key].shape == cnn_lstm_checkpoint["state_dict"][cnn_lstm_key].shape, f"shape mismatch for {cnn_lstm_key}"
# cnn_lstm_loaded.load_state_dict(cnn_lstm_checkpoint["state_dict"])
# cnn_lstm_loaded.to(DEVICE).eval()
# print("loaded", cnn_lstm_checkpoint["model_name"], "| parameters:", count_parameters(cnn_lstm_loaded))

## 13.15 Inference Using the Loaded Model
The reloaded network must reproduce the in-memory validation predictions.

In [ ]:
# cnn_lstm_loaded_val_pred = predict_network(cnn_lstm_loaded, x_val_t)
# assert np.allclose(cnn_lstm_loaded_val_pred, cnn_lstm_final_val_pred, atol=1e-3), "loaded CNN-LSTM model does not reproduce its predictions"
# print("loaded model reproduces", len(cnn_lstm_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(cnn_lstm_loaded_val_pred)["tail"]["rmse"], 3))

# 14. Temporal Convolutional Network (TCN)
A TCN stacks **dilated causal convolutions** with residual connections. It sees a long history with few layers, trains in parallel (no recurrence), and never looks at later cycles.

## 14.1 Data Preparation
Same tensors and loader settings as every sequence model. The check confirms the model input will be `(batch, window_size, features)`.

In [ ]:
tcn_batch_x, tcn_batch_y = next(iter(make_train_loader(x_train_t, y_train_t)))
assert tcn_batch_x.shape[1:] == (WINDOW_SIZE, n_features), "unexpected batch shape"
print("input batch:", tuple(tcn_batch_x.shape), "| targets:", tuple(tcn_batch_y.shape))

## 14.2 Network Architecture
Each residual block holds two causal convolutions with dropout and a shortcut. Block *i* uses dilation `2**i`, so the receptive field grows quickly. The output at the **last** time step feeds the dense head.

`(B, T, F)` -> transpose -> `(B, F, T)` -> blocks -> `(B, channels[-1], T)` -> last step -> head -> `(B,)`. The classes are defined in the next three steps.

## 14.3 Causal Convolution
A convolution is **causal** when the output at time *t* depends only on inputs at times <= *t*. This is done by padding only the **left** side by `(kernel - 1) * dilation`, so the output keeps the window length and no future cycle leaks into an earlier output.

In [ ]:
class CausalConv1d(nn.Module):
    """1D convolution that only sees the current and earlier time steps."""

    def __init__(self, in_channels, out_channels, kernel, dilation):
        super().__init__()
        self.left_padding = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel, dilation=dilation)

    def forward(self, x):                                  # x: (batch, channels, time)
        return self.conv(F.pad(x, (self.left_padding, 0)))  # pad the past side only -> same length, no future leakage


print("causal convolution defined")

## 14.4 Dilated Convolution
Dilation skips inputs between kernel taps. With dilations 1, 2, 4, ... the receptive field after *n* blocks of two convolutions is `1 + 2 * (kernel - 1) * (2**n - 1)` cycles. If it is shorter than the window, the network cannot use the oldest cycles.

In [ ]:
for tcn_config in DL_CONFIGS["TCN"]:
    tcn_blocks = len(tcn_config["channels"])
    tcn_field = 1 + 2 * (tcn_config["kernel"] - 1) * sum(2 ** i for i in range(tcn_blocks))
    print(f"channels {tcn_config['channels']}, kernel {tcn_config['kernel']}: receptive field {tcn_field} cycles (window {WINDOW_SIZE})")

## 14.5 Residual Connections
The shortcut adds the block input to its output, which keeps gradients healthy in deeper stacks. When the channel count changes, a 1x1 convolution makes the shapes match. The same cell assembles the full network.

In [ ]:
class TemporalBlock(nn.Module):
    """Two causal convolutions with ReLU and dropout, plus a residual shortcut."""

    def __init__(self, in_channels, out_channels, kernel, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel, dilation)
        self.dropout = nn.Dropout(dropout)
        self.shortcut = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        z = self.dropout(F.relu(self.conv1(x)))
        z = self.dropout(F.relu(self.conv2(z)))
        return F.relu(z + self.shortcut(x))  # residual connection


class TCNRegressor(nn.Module):
    """Stack of temporal blocks with dilation 1, 2, 4, ... and a dense head on the last time step."""

    def __init__(self, input_dim, channels, kernel, dropout):
        super().__init__()
        blocks, in_channels = [], input_dim
        for level, out_channels in enumerate(channels):
            blocks.append(TemporalBlock(in_channels, out_channels, kernel, 2 ** level, dropout))
            in_channels = out_channels
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Linear(channels[-1], 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def sequence_features(self, x):
        return self.blocks(x.transpose(1, 2))                   # (batch, channels[-1], window)

    def forward(self, x):
        return self.head(self.sequence_features(x)[:, :, -1]).squeeze(-1)  # last cycle only -> (batch,)


print("TCN defined")

## 14.6 Model Initialization
Create the network with the **default** configuration (first entry in `DL_CONFIGS`) and push one batch through it to confirm the output shape `(batch,)`.

In [ ]:
set_seed(RANDOM_STATE)
tcn_default_config = DL_CONFIGS["TCN"][0]
tcn_model = build_model("TCN", n_features, tcn_default_config)
with torch.no_grad():
    tcn_sample_output = tcn_model.eval()(x_train_t[:8])
assert tcn_sample_output.shape == (8,), f"expected output shape (8,), got {tuple(tcn_sample_output.shape)}"
print(tcn_model)
print("trainable parameters:", count_parameters(tcn_model), "| config:", tcn_default_config)

## 14.7 Causality Check
Proof that the network is causal: change only the **last 5 cycles** of a window and compare the internal features. Everything before those cycles must stay identical.

In [ ]:
tcn_model.eval()
tcn_original = x_train_t[:4].clone()
tcn_modified = tcn_original.clone()
tcn_modified[:, -5:, :] = torch.randn_like(tcn_modified[:, -5:, :])  # alter only the most recent cycles
with torch.no_grad():
    tcn_features_a = tcn_model.sequence_features(tcn_original)
    tcn_features_b = tcn_model.sequence_features(tcn_modified)
assert torch.allclose(tcn_features_a[:, :, :-5], tcn_features_b[:, :, :-5], atol=1e-6), "earlier outputs changed: the network is not causal"
assert not torch.allclose(tcn_features_a[:, :, -5:], tcn_features_b[:, :, -5:]), "recent outputs should react to recent inputs"
print("causality verified: earlier time steps ignore later inputs")

## 14.8 Loss Function and Optimizer
Mean squared error on targets scaled to `RUL / RUL_CAP`: it matches the RMSE metric and keeps the loss near 0 to 1. Adam adapts the step size per weight, and a small weight decay discourages overfitting. Gradients are clipped inside `train_network`.

In [ ]:
tcn_loss = nn.MSELoss()
tcn_optimizer = torch.optim.Adam(tcn_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
print("loss: MSE | optimizer: Adam | learning rate:", LEARNING_RATE)

## 14.9 Training Loop
Each epoch: forward pass, loss, backward pass, gradient clipping, optimizer step. After every epoch the model is scored on the validation tail (the epoch-level validation of the next step).

In [ ]:
tcn_loader = make_train_loader(x_train_t, y_train_t)
tcn_history = train_network(tcn_model, tcn_optimizer, tcn_loss, tcn_loader, EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail)
print("epochs run:", len(tcn_history["epoch"]), "| best validation-tail RMSE:", round(tcn_history["best_rmse"], 3))

## 14.10 Epoch-Level Validation
Training loss should fall steadily. Validation RMSE that stops improving while the training loss keeps falling means overfitting.

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=tcn_history["epoch"], y=tcn_history["train_loss"], mode="lines+markers", name="train loss (scaled MSE)"), secondary_y=False)
fig.add_trace(go.Scatter(x=tcn_history["epoch"], y=tcn_history["val_rmse"], mode="lines+markers", name="validation-tail RMSE (cycles)"), secondary_y=True)
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="train loss (scaled MSE)", secondary_y=False)
fig.update_yaxes(title_text="validation-tail RMSE (cycles)", secondary_y=True)
fig.update_layout(height=380, width=800, title_text="TCN: learning curves")
fig.show()

## 14.11 Early Stopping
Training stops when the validation-tail RMSE has not improved for `EARLY_STOPPING_PATIENCE` epochs, and the weights of the best epoch are restored. This is the cheapest protection against overfitting.

In [ ]:
tcn_stopped_early = len(tcn_history["epoch"]) < EPOCHS
print("stopped early:", tcn_stopped_early, "| epochs run:", len(tcn_history["epoch"]), "| best epoch:", tcn_history["best_epoch"])
assert 1 <= tcn_history["best_epoch"] <= len(tcn_history["epoch"]), "no valid best epoch was recorded"

## 14.12 Prediction and Inference
Inference uses `model.eval()` (dropout and batch norm in inference mode) and `torch.no_grad()` (no gradient bookkeeping). The output is rescaled to cycles.

In [ ]:
tcn_val_pred = predict_network(tcn_model, x_val_t)
assert tcn_val_pred.shape == y_val.shape and np.isfinite(tcn_val_pred).all(), "invalid validation predictions"
print("validation predictions:", tcn_val_pred.shape, "| range", round(float(tcn_val_pred.min()), 1), "to", round(float(tcn_val_pred.max()), 1))

## 14.13 Model Evaluation
Validation metrics on the test-like tail and on all rows, plus predicted versus true RUL for the tail.

In [ ]:
tcn_scores = evaluate_validation(tcn_val_pred)
display(scores_table({"TCN (default config)": tcn_scores}))
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_val_tail, y=tcn_val_pred[val_tail_mask], mode="markers", marker=dict(size=4, opacity=0.4), name="validation tail"))
fig.add_trace(go.Scatter(x=[0, RUL_CAP], y=[0, RUL_CAP], mode="lines", line=dict(dash="dash", color="black"), name="ideal"))
fig.update_layout(height=450, width=520, xaxis_title="true RUL", yaxis_title="predicted RUL", title_text="TCN: validation tail")
fig.show()

## 14.14 Hyperparameter Tuning
A small, fixed list of configurations (`DL_CONFIGS`) is trained for `TUNE_EPOCHS` epochs each, with the same seed, the same batches, and the **same validation split**. The best configuration is the one with the lowest validation-tail RMSE.

In [ ]:
tcn_trials = []
for tcn_trial_config in DL_CONFIGS["TCN"]:
    set_seed(RANDOM_STATE)
    tcn_trial_model = build_model("TCN", n_features, tcn_trial_config)
    tcn_trial_optimizer = torch.optim.Adam(tcn_trial_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    tcn_trial_history = train_network(tcn_trial_model, tcn_trial_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                       TUNE_EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    tcn_trials.append({"config": tcn_trial_config, "val_tail_rmse": tcn_trial_history["best_rmse"],
                       "best_epoch": tcn_trial_history["best_epoch"], "parameters": count_parameters(tcn_trial_model)})
tcn_trials_table = pd.DataFrame(tcn_trials).sort_values("val_tail_rmse")
tcn_trials_table.round(3)

## 14.15 Best Configuration Selection
If the best trial is not the default, that configuration is retrained for the full `EPOCHS` with early stopping. It replaces the default model only if its validation-tail RMSE is lower.

In [ ]:
tcn_best_config = tcn_trials_table.iloc[0]["config"]
tcn_final_model, tcn_final_config, tcn_final_history = tcn_model, tcn_default_config, tcn_history
if tcn_best_config != tcn_default_config:
    set_seed(RANDOM_STATE)
    tcn_candidate = build_model("TCN", n_features, tcn_best_config)
    tcn_candidate_optimizer = torch.optim.Adam(tcn_candidate.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    tcn_candidate_history = train_network(tcn_candidate, tcn_candidate_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                           EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    if tcn_candidate_history["best_rmse"] < tcn_history["best_rmse"]:
        tcn_final_model, tcn_final_config, tcn_final_history = tcn_candidate, tcn_best_config, tcn_candidate_history
print("selected config:", tcn_final_config, "| validation-tail RMSE:", round(tcn_final_history["best_rmse"], 3), "| best epoch:", tcn_final_history["best_epoch"])

## 14.16 Validation Result of the Selected Model
The selected network is scored once more and stored for the comparison in section 16.

In [ ]:
tcn_final_val_pred = predict_network(tcn_final_model, x_val_t)
tcn_final_scores = evaluate_validation(tcn_final_val_pred)
validation_results["TCN"] = {"tail": tcn_final_scores["tail"], "all": tcn_final_scores["all"], "family": "nn",
                            "params": tcn_final_config, "best_epoch": tcn_final_history["best_epoch"],
                            "parameters": count_parameters(tcn_final_model)}
scores_table({"TCN (selected)": tcn_final_scores})

## 14.17 Saving Model Artifacts
Weights (as plain tensors), the architecture configuration, the input size, the feature names, the window size, the RUL cap, and the name of the preprocessing file go into one checkpoint. Together with `preprocessor_*.joblib` this is everything needed to reproduce predictions.

In [ ]:
tcn_st_path = ARTIFACT_PATH / f"tcn_{DATASET_ID.lower()}.safetensors"
save_file({key: value.detach().cpu().contiguous() for key, value in tcn_final_model.state_dict().items()},
          str(tcn_st_path),
          metadata={"model_name": "TCN", "config": json.dumps(tcn_final_config), "input_dim": str(n_features),
                    "feature_names": json.dumps(live_sensors), "window_size": str(WINDOW_SIZE), "rul_cap": str(RUL_CAP),
                    "preprocessor_file": preprocessor_path.name})
print("saved", tcn_st_path.name, f"({tcn_st_path.stat().st_size / 1024:.0f} KB)")

## 14.18 Loading the Saved Model
The architecture is rebuilt from the stored configuration and **checked against the stored weights** (same parameter names and shapes) before the weights are loaded.

In [ ]:
# tcn_checkpoint = torch.load(tcn_path, map_location="cpu", weights_only=True)
# assert tcn_checkpoint["model_name"] == "TCN" and tcn_checkpoint["window_size"] == WINDOW_SIZE and tcn_checkpoint["rul_cap"] == RUL_CAP
# tcn_loaded = build_model("TCN", tcn_checkpoint["input_dim"], tcn_checkpoint["config"])
# tcn_expected_state = tcn_loaded.state_dict()
# assert list(tcn_expected_state.keys()) == list(tcn_checkpoint["state_dict"].keys()), "parameter names differ from the checkpoint"
# for tcn_key in tcn_expected_state:
#     assert tcn_expected_state[tcn_key].shape == tcn_checkpoint["state_dict"][tcn_key].shape, f"shape mismatch for {tcn_key}"
# tcn_loaded.load_state_dict(tcn_checkpoint["state_dict"])
# tcn_loaded.to(DEVICE).eval()
# print("loaded", tcn_checkpoint["model_name"], "| parameters:", count_parameters(tcn_loaded))

## 14.19 Inference Using the Loaded Model
The reloaded network must reproduce the in-memory validation predictions.

In [ ]:
# tcn_loaded_val_pred = predict_network(tcn_loaded, x_val_t)
# assert np.allclose(tcn_loaded_val_pred, tcn_final_val_pred, atol=1e-3), "loaded TCN model does not reproduce its predictions"
# print("loaded model reproduces", len(tcn_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(tcn_loaded_val_pred)["tail"]["rmse"], 3))

# 15. Temporal Transformer Regression
Self-attention lets every cycle in the window look at **every other cycle** directly, so distant events can matter as much as recent ones. Attention has no sense of order, so a positional encoding is added.

## 15.1 Data Preparation
Same tensors and loader settings as every sequence model. The check confirms the model input will be `(batch, window_size, features)`.

In [ ]:
transformer_batch_x, transformer_batch_y = next(iter(make_train_loader(x_train_t, y_train_t)))
assert transformer_batch_x.shape[1:] == (WINDOW_SIZE, n_features), "unexpected batch shape"
print("input batch:", tuple(transformer_batch_x.shape), "| targets:", tuple(transformer_batch_y.shape))

## 15.2 Input Projection
A linear layer maps the `features` sensors of each cycle to an embedding of size `d_model`: `(B, T, F)` -> `(B, T, d_model)`.

In [ ]:
projection_demo = nn.Linear(n_features, DL_CONFIGS["Transformer"][0]["d_model"])
with torch.no_grad():
    print("projection:", tuple(x_train_t[:2].shape), "->", tuple(projection_demo(x_train_t[:2]).shape))

## 15.3 Positional Encoding
Fixed sinusoids of different frequencies are added to the embeddings so the model knows the order of the cycles. The table has exactly `WINDOW_SIZE` positions (stored as a buffer, so it is saved with the weights).

In [ ]:
class PositionalEncoding(nn.Module):
    """Fixed sinusoidal position table added to the embedded window."""

    def __init__(self, d_model, max_len):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)                       # (max_len, 1)
        frequencies = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        table = torch.zeros(1, max_len, d_model)
        table[0, :, 0::2] = torch.sin(position * frequencies)
        table[0, :, 1::2] = torch.cos(position * frequencies)
        self.register_buffer("table", table)

    def forward(self, x):                                   # x: (batch, window, d_model)
        assert x.size(1) <= self.table.size(1), "window is longer than the positional table"
        return x + self.table[:, :x.size(1)]


demo_encoding = PositionalEncoding(DL_CONFIGS["Transformer"][0]["d_model"], WINDOW_SIZE)
fig = go.Figure(go.Heatmap(z=demo_encoding.table[0].numpy().T, colorscale="RdBu_r", colorbar=dict(title="value")))
fig.update_layout(height=320, width=700, xaxis_title="cycle in window", yaxis_title="embedding dimension", title_text="Sinusoidal positional encoding")
fig.show()

## 15.4 Transformer Encoder Architecture
Projection, positional encoding, a stack of encoder layers (multi-head self-attention plus a feed-forward block), then the output of the **last** cycle goes to the head. `d_model` must be divisible by `nhead` (each head gets `d_model / nhead` dimensions) and even (sine/cosine pairs).

In [ ]:
class TransformerRegressor(nn.Module):
    """Linear projection + positional encoding + Transformer encoder + dense head on the last cycle."""

    def __init__(self, input_dim, d_model, nhead, layers, ff, dropout, max_len):
        super().__init__()
        assert d_model % nhead == 0 and d_model % 2 == 0, "d_model must be even and divisible by nhead"
        self.projection = nn.Linear(input_dim, d_model)
        self.positional = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):                                # x: (batch, window, features)
        z = self.positional(self.projection(x))          # (batch, window, d_model)
        z = self.encoder(z)                              # (batch, window, d_model)
        return self.head(z[:, -1, :]).squeeze(-1)        # last cycle only -> (batch,)


print("Transformer defined")

## 15.5 Regression Output Layer
The head turns the `d_model` vector of the last cycle into one number (RUL / RUL_CAP). The check also confirms that every configuration in the search has compatible embedding size and head count.

In [ ]:
for transformer_config in DL_CONFIGS["Transformer"]:
    assert transformer_config["d_model"] % transformer_config["nhead"] == 0 and transformer_config["d_model"] % 2 == 0, f"incompatible config {transformer_config}"
transformer_demo = TransformerRegressor(n_features, max_len=WINDOW_SIZE, **DL_CONFIGS["Transformer"][0])
print(transformer_demo.head)
print("all Transformer configurations have compatible d_model / nhead")

## 15.6 Model Initialization
Create the network with the **default** configuration (first entry in `DL_CONFIGS`) and push one batch through it to confirm the output shape `(batch,)`.

In [ ]:
set_seed(RANDOM_STATE)
transformer_default_config = DL_CONFIGS["Transformer"][0]
transformer_model = build_model("Transformer", n_features, transformer_default_config)
with torch.no_grad():
    transformer_sample_output = transformer_model.eval()(x_train_t[:8])
assert transformer_sample_output.shape == (8,), f"expected output shape (8,), got {tuple(transformer_sample_output.shape)}"
print(transformer_model)
print("trainable parameters:", count_parameters(transformer_model), "| config:", transformer_default_config)

## 15.7 Loss Function and Optimizer
Mean squared error on targets scaled to `RUL / RUL_CAP`: it matches the RMSE metric and keeps the loss near 0 to 1. Adam adapts the step size per weight, and a small weight decay discourages overfitting. Gradients are clipped inside `train_network`.

In [ ]:
transformer_loss = nn.MSELoss()
transformer_optimizer = torch.optim.Adam(transformer_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
print("loss: MSE | optimizer: Adam | learning rate:", LEARNING_RATE)

## 15.8 Training Loop
Each epoch: forward pass, loss, backward pass, gradient clipping, optimizer step. After every epoch the model is scored on the validation tail (the epoch-level validation of the next step).

In [ ]:
transformer_loader = make_train_loader(x_train_t, y_train_t)
transformer_history = train_network(transformer_model, transformer_optimizer, transformer_loss, transformer_loader, EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail)
print("epochs run:", len(transformer_history["epoch"]), "| best validation-tail RMSE:", round(transformer_history["best_rmse"], 3))

## 15.9 Epoch-Level Validation
Training loss should fall steadily. Validation RMSE that stops improving while the training loss keeps falling means overfitting.

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=transformer_history["epoch"], y=transformer_history["train_loss"], mode="lines+markers", name="train loss (scaled MSE)"), secondary_y=False)
fig.add_trace(go.Scatter(x=transformer_history["epoch"], y=transformer_history["val_rmse"], mode="lines+markers", name="validation-tail RMSE (cycles)"), secondary_y=True)
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="train loss (scaled MSE)", secondary_y=False)
fig.update_yaxes(title_text="validation-tail RMSE (cycles)", secondary_y=True)
fig.update_layout(height=380, width=800, title_text="Transformer: learning curves")
fig.show()

## 15.10 Early Stopping
Training stops when the validation-tail RMSE has not improved for `EARLY_STOPPING_PATIENCE` epochs, and the weights of the best epoch are restored. This is the cheapest protection against overfitting.

In [ ]:
transformer_stopped_early = len(transformer_history["epoch"]) < EPOCHS
print("stopped early:", transformer_stopped_early, "| epochs run:", len(transformer_history["epoch"]), "| best epoch:", transformer_history["best_epoch"])
assert 1 <= transformer_history["best_epoch"] <= len(transformer_history["epoch"]), "no valid best epoch was recorded"

## 15.11 Prediction and Inference
Inference uses `model.eval()` (dropout and batch norm in inference mode) and `torch.no_grad()` (no gradient bookkeeping). The output is rescaled to cycles.

In [ ]:
transformer_val_pred = predict_network(transformer_model, x_val_t)
assert transformer_val_pred.shape == y_val.shape and np.isfinite(transformer_val_pred).all(), "invalid validation predictions"
print("validation predictions:", transformer_val_pred.shape, "| range", round(float(transformer_val_pred.min()), 1), "to", round(float(transformer_val_pred.max()), 1))

## 15.12 Model Evaluation
Validation metrics on the test-like tail and on all rows, plus predicted versus true RUL for the tail.

In [ ]:
transformer_scores = evaluate_validation(transformer_val_pred)
display(scores_table({"Transformer (default config)": transformer_scores}))
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_val_tail, y=transformer_val_pred[val_tail_mask], mode="markers", marker=dict(size=4, opacity=0.4), name="validation tail"))
fig.add_trace(go.Scatter(x=[0, RUL_CAP], y=[0, RUL_CAP], mode="lines", line=dict(dash="dash", color="black"), name="ideal"))
fig.update_layout(height=450, width=520, xaxis_title="true RUL", yaxis_title="predicted RUL", title_text="Transformer: validation tail")
fig.show()

## 15.13 Hyperparameter Tuning
A small, fixed list of configurations (`DL_CONFIGS`) is trained for `TUNE_EPOCHS` epochs each, with the same seed, the same batches, and the **same validation split**. The best configuration is the one with the lowest validation-tail RMSE.

In [ ]:
transformer_trials = []
for transformer_trial_config in DL_CONFIGS["Transformer"]:
    set_seed(RANDOM_STATE)
    transformer_trial_model = build_model("Transformer", n_features, transformer_trial_config)
    transformer_trial_optimizer = torch.optim.Adam(transformer_trial_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    transformer_trial_history = train_network(transformer_trial_model, transformer_trial_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                       TUNE_EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    transformer_trials.append({"config": transformer_trial_config, "val_tail_rmse": transformer_trial_history["best_rmse"],
                       "best_epoch": transformer_trial_history["best_epoch"], "parameters": count_parameters(transformer_trial_model)})
transformer_trials_table = pd.DataFrame(transformer_trials).sort_values("val_tail_rmse")
transformer_trials_table.round(3)

## 15.14 Best Configuration Selection
If the best trial is not the default, that configuration is retrained for the full `EPOCHS` with early stopping. It replaces the default model only if its validation-tail RMSE is lower.

In [ ]:
transformer_best_config = transformer_trials_table.iloc[0]["config"]
transformer_final_model, transformer_final_config, transformer_final_history = transformer_model, transformer_default_config, transformer_history
if transformer_best_config != transformer_default_config:
    set_seed(RANDOM_STATE)
    transformer_candidate = build_model("Transformer", n_features, transformer_best_config)
    transformer_candidate_optimizer = torch.optim.Adam(transformer_candidate.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    transformer_candidate_history = train_network(transformer_candidate, transformer_candidate_optimizer, nn.MSELoss(), make_train_loader(x_train_t, y_train_t),
                                           EPOCHS, EARLY_STOPPING_PATIENCE, x_val_tail_t, y_val_tail, verbose=False)
    if transformer_candidate_history["best_rmse"] < transformer_history["best_rmse"]:
        transformer_final_model, transformer_final_config, transformer_final_history = transformer_candidate, transformer_best_config, transformer_candidate_history
print("selected config:", transformer_final_config, "| validation-tail RMSE:", round(transformer_final_history["best_rmse"], 3), "| best epoch:", transformer_final_history["best_epoch"])

## 15.15 Validation Result of the Selected Model
The selected network is scored once more and stored for the comparison in section 16.

In [ ]:
transformer_final_val_pred = predict_network(transformer_final_model, x_val_t)
transformer_final_scores = evaluate_validation(transformer_final_val_pred)
validation_results["Transformer"] = {"tail": transformer_final_scores["tail"], "all": transformer_final_scores["all"], "family": "nn",
                            "params": transformer_final_config, "best_epoch": transformer_final_history["best_epoch"],
                            "parameters": count_parameters(transformer_final_model)}
scores_table({"Transformer (selected)": transformer_final_scores})

## 15.16 Saving Model Artifacts
Weights (as plain tensors), the architecture configuration, the input size, the feature names, the window size, the RUL cap, and the name of the preprocessing file go into one checkpoint. Together with `preprocessor_*.joblib` this is everything needed to reproduce predictions.

In [ ]:
transformer_st_path = ARTIFACT_PATH / f"temporal_transformer_{DATASET_ID.lower()}.safetensors"
save_file({key: value.detach().cpu().contiguous() for key, value in transformer_final_model.state_dict().items()},
          str(transformer_st_path),
          metadata={"model_name": "Transformer", "config": json.dumps(transformer_final_config), "input_dim": str(n_features),
                    "feature_names": json.dumps(live_sensors), "window_size": str(WINDOW_SIZE), "rul_cap": str(RUL_CAP),
                    "preprocessor_file": preprocessor_path.name})
print("saved", transformer_st_path.name, f"({transformer_st_path.stat().st_size / 1024:.0f} KB)")

## 15.17 Loading the Saved Model
The architecture is rebuilt from the stored configuration and **checked against the stored weights** (same parameter names and shapes) before the weights are loaded.

In [ ]:
# transformer_checkpoint = torch.load(transformer_path, map_location="cpu", weights_only=True)
# assert transformer_checkpoint["model_name"] == "Transformer" and transformer_checkpoint["window_size"] == WINDOW_SIZE and transformer_checkpoint["rul_cap"] == RUL_CAP
# transformer_loaded = build_model("Transformer", transformer_checkpoint["input_dim"], transformer_checkpoint["config"])
# transformer_expected_state = transformer_loaded.state_dict()
# assert list(transformer_expected_state.keys()) == list(transformer_checkpoint["state_dict"].keys()), "parameter names differ from the checkpoint"
# for transformer_key in transformer_expected_state:
#     assert transformer_expected_state[transformer_key].shape == transformer_checkpoint["state_dict"][transformer_key].shape, f"shape mismatch for {transformer_key}"
# transformer_loaded.load_state_dict(transformer_checkpoint["state_dict"])
# transformer_loaded.to(DEVICE).eval()
# print("loaded", transformer_checkpoint["model_name"], "| parameters:", count_parameters(transformer_loaded))

## 15.18 Inference Using the Loaded Model
The reloaded network must reproduce the in-memory validation predictions.

In [ ]:
# transformer_loaded_val_pred = predict_network(transformer_loaded, x_val_t)
# assert np.allclose(transformer_loaded_val_pred, transformer_final_val_pred, atol=1e-3), "loaded Transformer model does not reproduce its predictions"
# print("loaded model reproduces", len(transformer_loaded_val_pred), "validation predictions | tail RMSE", round(evaluate_validation(transformer_loaded_val_pred)["tail"]["rmse"], 3))

# 16. Model Comparison and Selection
All six models are compared on the **validation** partition (engines never used for fitting). The official test labels play no role in tuning, feature selection, or choosing the architecture.


## 16.1 Consolidated Validation Table

In [ ]:
model_order = ["Random Forest", "XGBoost", "LSTM", "CNN-LSTM", "TCN", "Transformer"]
missing_models = [name for name in model_order if name not in validation_results]
assert not missing_models, f"these models have no validation result: {missing_models}"
comparison_table = scores_table({name: validation_results[name] for name in model_order}).sort_values("tail RMSE")
comparison_table

## 16.2 Selection Criterion
The winner is the model with the lowest **validation-tail RMSE**: the tail has the same RUL range as the test engines (which stop shortly before failure), and RMSE is the metric the models were trained to reduce. NASA score per sample and MAE are shown as tie-breakers for the reader. The plateau rows ("all" columns) are easier and are not used for the decision.

In [ ]:
selected_name = comparison_table.index[0]
selected_result = validation_results[selected_name]
bar_colors = ["seagreen" if name == selected_name else "gray" for name in comparison_table.index]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Validation-tail RMSE (cycles)", "Validation-tail NASA score per sample"))
fig.add_trace(go.Bar(x=list(comparison_table.index), y=comparison_table["tail RMSE"], marker_color=bar_colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=list(comparison_table.index), y=comparison_table["tail NASA/sample"], marker_color=bar_colors, showlegend=False), row=1, col=2)
fig.update_layout(height=380, width=1000, title_text=f"{DATASET_ID}: validation comparison (selected model in green)")
fig.show()
print(f"selected model: {selected_name} | validation-tail RMSE {selected_result['tail']['rmse']:.3f} | settings: {selected_result['params']}")

# 17. Final Training and Test Evaluation


## 17.1 Final Training Configuration
The selected architecture and its validation-selected hyperparameters are trained again on **all** official training engines (the validation engines join the training data, since no further selection happens). A network has no validation data left for early stopping, so it trains for the best epoch count found in section 12-15.

In [ ]:
final_family = selected_result["family"]
final_params = selected_result["params"]
final_epochs = selected_result.get("best_epoch", None)
print("final model:", selected_name, "| family:", final_family)
print("hyperparameters:", final_params)
print("training epochs (neural networks only):", final_epochs)

## 17.2 Full Training-Data Preparation
Every learned preprocessing step is **refit on all official training engines**: imputer, operating-condition model, sensor selection, and scalers. The test set is still only transformed, never fitted.

In [ ]:
full_train_df = train_df.copy()
final_imputer = SimpleImputer(strategy="median").fit(full_train_df[measurement_cols])
full_train_df[measurement_cols] = final_imputer.transform(full_train_df[measurement_cols])
final_test_df = test_raw_df.copy()
final_test_df = final_test_df.sort_values(["unit", "cycle"]).reset_index(drop=True)
final_test_df[measurement_cols] = final_test_df[measurement_cols].astype("float64").replace([np.inf, -np.inf], np.nan)
final_test_df[measurement_cols] = final_imputer.transform(final_test_df[measurement_cols])
print("imputer refit on", len(full_train_df), "rows | engines:", full_train_df["unit"].nunique())

Refit the operating-condition model on all official training engines and assign condition labels.

In [ ]:
if n_conditions == 1:
    final_settings_scaler, final_kmeans = None, None
else:
    final_settings_scaler = StandardScaler().fit(full_train_df[SETTING_COLS])
    final_kmeans = KMeans(n_clusters=n_conditions, n_init=10, random_state=RANDOM_STATE)
    final_kmeans.fit(final_settings_scaler.transform(full_train_df[SETTING_COLS]))
full_train_df["condition"] = assign_conditions(full_train_df, final_settings_scaler, final_kmeans)
final_test_df["condition"] = assign_conditions(final_test_df, final_settings_scaler, final_kmeans)
assert set(full_train_df["condition"].unique()) == set(range(n_conditions)), "a condition is empty in the full training data"
print("conditions refit | rows per condition:", np.bincount(full_train_df["condition"]).tolist())

Build the RUL labels of the full training data and recompute the sensor selection on it.

In [ ]:
full_train_df = add_rul_labels(full_train_df, RUL_CAP)
final_std = full_train_df.groupby("condition")[SENSOR_COLS].std().fillna(0.0)
final_unique = full_train_df.groupby("condition")[SENSOR_COLS].nunique()
final_flat = ((final_std < VARIANCE_THRESHOLD) | (final_unique <= MIN_UNIQUE_VALUES)).all(axis=0)
final_live_sensors = [c for c in SENSOR_COLS if not final_flat[c]]
assert len(final_live_sensors) > 0, "every sensor was flagged flat"
print("live sensors (full training data):", len(final_live_sensors), "| same as validation experiment:", final_live_sensors == live_sensors)

Refit the per-condition scalers (and the global fallback) on all official training rows.

In [ ]:
final_base_cols = ["unit", "cycle"] + SETTING_COLS + ["condition"] + final_live_sensors
final_train_sel = full_train_df[final_base_cols + ["RUL_linear", "RUL"]].copy()
final_test_sel = final_test_df[final_base_cols].copy()
final_condition_scalers = {}
for condition in sorted(final_train_sel["condition"].unique()):
    rows = final_train_sel.loc[final_train_sel["condition"] == condition, final_live_sensors]
    if len(rows) >= MIN_CLUSTER_ROWS:
        final_condition_scalers[int(condition)] = StandardScaler().fit(rows)
final_fallback_scaler = StandardScaler().fit(final_train_sel[final_live_sensors])
print("scalers refit | per-condition:", sorted(final_condition_scalers), "| fallback rows:", len(final_train_sel))

Transform the full training data and the test set with the refit scalers.

In [ ]:
final_train_prepared = normalize_sensors(final_train_sel, final_live_sensors, final_condition_scalers, final_fallback_scaler)
final_test_prepared = normalize_sensors(final_test_sel, final_live_sensors, final_condition_scalers, final_fallback_scaler)
assert np.isfinite(final_train_prepared[final_live_sensors].to_numpy()).all() and np.isfinite(final_test_prepared[final_live_sensors].to_numpy()).all()
assert final_fallback_scaler.n_samples_seen_ == len(full_train_df), "final scaler must see exactly the training rows"
print("normalized | train", final_train_prepared.shape, "| test", final_test_prepared.shape)

## 17.3 Final Feature and Sequence Preparation
Training windows use every cycle of every official training engine. The test set uses **only the final window of each engine**: the model sees only what was recorded up to the last available cycle.

In [ ]:
final_train_windows, final_train_units, final_train_cycles = build_windows(final_train_prepared, final_live_sensors, WINDOW_SIZE)
final_test_windows, final_test_units, final_test_cycles = build_windows(final_test_prepared, final_live_sensors, WINDOW_SIZE, last_only=True)
final_y_train = final_train_prepared["RUL"].to_numpy(dtype=np.float32)
assert final_train_windows.shape == (len(final_train_prepared), WINDOW_SIZE, len(final_live_sensors))
assert final_test_windows.shape == (final_test_prepared["unit"].nunique(), WINDOW_SIZE, len(final_live_sensors))
assert len(final_y_train) == len(final_train_windows)
print("train windows", final_train_windows.shape, "| test windows", final_test_windows.shape)

Tabular features of the final windows (used only when a tree model is selected).

In [ ]:
final_extra_names = ["cycle"] + (["condition"] if n_conditions > 1 else [])
final_test_last_rows = final_test_prepared.groupby("unit").tail(1)
final_X_train, final_feature_names = extract_window_features(final_train_windows, final_train_prepared[final_extra_names].to_numpy(), final_extra_names, final_live_sensors)
final_X_test, _ = extract_window_features(final_test_windows, final_test_last_rows[final_extra_names].to_numpy(), final_extra_names, final_live_sensors)
final_X_train_df = pd.DataFrame(final_X_train, columns=final_feature_names)
final_X_test_df = pd.DataFrame(final_X_test, columns=final_feature_names)
assert np.isfinite(final_X_train).all() and np.isfinite(final_X_test).all() and not any("RUL" in n for n in final_feature_names)
print("tabular features:", final_X_train_df.shape, final_X_test_df.shape, "(used when a tree model is selected)")

## 17.4 Final Model Training
Trains the selected family with the stored hyperparameters on all official training engines.

In [ ]:
set_seed(RANDOM_STATE)
if selected_name == "Random Forest":
    final_model = RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE, **final_params)
    final_model.fit(final_X_train_df, final_y_train)
elif selected_name == "XGBoost":
    final_model = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, **final_params)
    final_model.fit(final_X_train_df, final_y_train)
else:
    final_input_dim = len(final_live_sensors)
    final_model = build_model(selected_name, final_input_dim, final_params)
    final_optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    final_loader = make_train_loader(torch.from_numpy(final_train_windows), torch.from_numpy(final_y_train.copy()))
    final_history = train_network(final_model, final_optimizer, nn.MSELoss(), final_loader, int(final_epochs), verbose=False)  # fixed epoch count, no validation data left
    print("final training loss per epoch (last 3):", [round(v, 5) for v in final_history["train_loss"][-3:]])
print("final", selected_name, "trained on", len(final_y_train), "rows from", full_train_df["unit"].nunique(), "engines")

## 17.5 Final Model Persistence
The model and **all refit preprocessing components** needed for inference are saved together.

In [ ]:
final_preprocessor = {
    "dataset_id": DATASET_ID, "measurement_cols": measurement_cols, "imputer": final_imputer,
    "n_conditions": n_conditions, "settings_scaler": final_settings_scaler, "kmeans": final_kmeans,
    "live_sensors": final_live_sensors, "condition_scalers": final_condition_scalers, "fallback_scaler": final_fallback_scaler,
    "window_size": WINDOW_SIZE, "rul_cap": RUL_CAP, "base_cols": final_base_cols,
}
final_preprocessor_path = ARTIFACT_PATH / f"final_preprocessor_{DATASET_ID.lower()}.joblib"
joblib.dump(final_preprocessor, final_preprocessor_path)
final_stem = f"final_{selected_name.lower().replace(' ', '_').replace('-', '_')}_{DATASET_ID.lower()}"
if selected_name == "Random Forest":
    final_model_path = ARTIFACT_PATH / f"{final_stem}.joblib"
    joblib.dump({"model": final_model, "feature_names": final_feature_names, "model_name": selected_name, "config": final_params}, final_model_path)
elif selected_name == "XGBoost":
    final_model_path = ARTIFACT_PATH / f"{final_stem}.json"
    final_model.save_model(final_model_path)
    joblib.dump({"feature_names": final_feature_names, "model_name": selected_name, "config": final_params}, ARTIFACT_PATH / f"{final_stem}_meta.joblib")
else:
   final_model_path = ARTIFACT_PATH / f"{final_stem}.safetensors"
   save_file({k: v.detach().cpu().contiguous() for k, v in final_model.state_dict().items()}, str(final_model_path),
             metadata={"model_name": selected_name, "config": json.dumps(final_params), "input_dim": str(len(final_live_sensors)),
                    "feature_names": json.dumps(final_live_sensors), "window_size": str(WINDOW_SIZE), "rul_cap": str(RUL_CAP),
                    "preprocessor_file": final_preprocessor_path.name})
print("saved:", final_model_path.name, "and", final_preprocessor_path.name)

## 17.6 Load the Final Model
Everything is rebuilt **from disk only**: from here on the notebook uses the loaded model and the loaded preprocessing, not the objects still in memory.

In [ ]:
loaded_preprocessor = joblib.load(final_preprocessor_path)
if selected_name == "Random Forest":
    loaded_bundle = joblib.load(final_model_path)
    loaded_final_model, loaded_feature_names = loaded_bundle["model"], loaded_bundle["feature_names"]
elif selected_name == "XGBoost":
    loaded_bundle = joblib.load(ARTIFACT_PATH / f"{final_stem}_meta.joblib")
    loaded_final_model = xgb.XGBRegressor()
    loaded_final_model.load_model(final_model_path)
    loaded_feature_names = loaded_bundle["feature_names"]
else:
    with safe_open(str(final_model_path), framework="pt", device="cpu") as f:
        meta = f.metadata()
        state_dict = {k: f.get_tensor(k) for k in f.keys()}

    assert (meta["model_name"] == selected_name
            and int(meta["window_size"]) == WINDOW_SIZE
            and float(meta["rul_cap"]) == float(RUL_CAP)), "metadata differs from the notebook settings"

    loaded_final_model = build_model(meta["model_name"], int(meta["input_dim"]), json.loads(meta["config"]))
    loaded_final_model.load_state_dict(state_dict, strict=True)  # also checks names and shapes
    loaded_final_model.to(DEVICE).eval()
    loaded_feature_names = json.loads(meta["feature_names"])
print("loaded:", selected_name, "| features:", len(loaded_feature_names))

A single prediction function serves the final evaluation, the trajectory plot, and the verification section. It starts from **windows** of the prepared data, so the model sees exactly what it saw in training.

In [ ]:
def predict_with_loaded_model(windows, extra_values):
    """RUL predictions (cycles) of the loaded final model for windows of shape (n, window_size, n_sensors)."""
    if selected_name in ("Random Forest", "XGBoost"):
        features, names = extract_window_features(windows, extra_values, final_extra_names, loaded_preprocessor["live_sensors"])
        assert names == loaded_feature_names, "feature names or order differ from training"
        return np.clip(loaded_final_model.predict(pd.DataFrame(features, columns=names)), 0.0, None)
    return predict_network(loaded_final_model, torch.from_numpy(np.ascontiguousarray(windows, dtype=np.float32)))


print("prediction function defined")

## 17.7 Test Data Inference
Start from the **raw** test file, apply the **loaded** preprocessing (imputer, conditions, sensor selection, scalers), take the final window of every engine, and predict. Exactly one prediction per test engine.

In [ ]:
inference_df = test_raw_df.sort_values(["unit", "cycle"]).reset_index(drop=True)
inference_df[measurement_cols] = loaded_preprocessor["imputer"].transform(inference_df[measurement_cols].astype("float64").replace([np.inf, -np.inf], np.nan))
inference_df["condition"] = assign_conditions(inference_df, loaded_preprocessor["settings_scaler"], loaded_preprocessor["kmeans"])
inference_df = normalize_sensors(inference_df[loaded_preprocessor["base_cols"]], loaded_preprocessor["live_sensors"],
                                 loaded_preprocessor["condition_scalers"], loaded_preprocessor["fallback_scaler"])
inference_windows, inference_units, inference_cycles = build_windows(inference_df, loaded_preprocessor["live_sensors"], WINDOW_SIZE, last_only=True)
inference_extra = inference_df.groupby("unit").tail(1)[final_extra_names].to_numpy()
test_predictions = predict_with_loaded_model(inference_windows, inference_extra)
assert test_predictions.shape == (len(test_rul_df),), f"expected {len(test_rul_df)} predictions, got {test_predictions.shape}"
assert np.isfinite(test_predictions).all(), "non-finite test predictions"
print("predictions:", test_predictions.shape, "| range", round(float(test_predictions.min()), 1), "to", round(float(test_predictions.max()), 1))

## 17.8 Ground-Truth Comparison
Test engine ids are 1..N (checked in 3.3), so row *i* of `RUL_{DATASET_ID}.txt` belongs to engine *i*. The official RUL stays **uncapped**, even though training targets were capped.

In [ ]:
assert np.array_equal(inference_units, np.arange(1, len(test_rul_df) + 1)), "test engines are not ordered like the RUL file"
test_true_rul = test_rul_df["true_rul"].to_numpy(dtype=float)
comparison_df = pd.DataFrame({"engine": inference_units, "true RUL": test_true_rul, "predicted RUL": test_predictions.round(1)})
comparison_df["error (pred - true)"] = (comparison_df["predicted RUL"] - comparison_df["true RUL"]).round(1)
comparison_df.head(10)

## 17.9 Final Evaluation Metrics
RMSE, MAE, and the NASA asymmetric score (`error = predicted - actual`; early: `exp(-error/13) - 1`, late: `exp(error/10) - 1`; summed over test engines).

In [ ]:
test_scores = evaluate_predictions(test_true_rul, test_predictions)
test_errors = test_predictions - test_true_rul
test_scores_table = pd.Series({"RMSE (cycles)": test_scores["rmse"], "MAE (cycles)": test_scores["mae"], "NASA score": test_scores["nasa"],
                               "late predictions (%)": 100 * float((test_errors > 0).mean())}, name=f"{selected_name} on {DATASET_ID} test")
assert abs(test_scores["nasa"] - nasa_score(test_true_rul, test_predictions)) < 1e-9
test_scores_table.round(3).to_frame()

## 17.10 Visualization
Predicted versus actual RUL, the scatter with its ideal line, the residual distribution, and the model comparison from section 16.

In [ ]:
order = np.argsort(test_true_rul)
limit = max(test_true_rul.max(), test_predictions.max()) * 1.05
fig = make_subplots(rows=1, cols=3, subplot_titles=("Predicted vs actual RUL", "Prediction scatter", "Residual distribution"))
fig.add_trace(go.Scatter(y=test_true_rul[order], mode="lines", name="actual", line=dict(color="black")), row=1, col=1)
fig.add_trace(go.Scatter(y=test_predictions[order], mode="lines", name="predicted", line=dict(color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=test_true_rul, y=test_predictions, mode="markers", name="engines", marker=dict(size=6, opacity=0.7)), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, limit], y=[0, limit], mode="lines", name="ideal", line=dict(dash="dash", color="black")), row=1, col=2)
fig.add_trace(go.Histogram(x=test_errors, nbinsx=20, name="residuals", marker_color="purple"), row=1, col=3)
fig.add_vline(x=0, line_dash="dash", row=1, col=3)
fig.update_xaxes(title_text="test engine (sorted by actual RUL)", row=1, col=1); fig.update_yaxes(title_text="RUL (cycles)", row=1, col=1)
fig.update_xaxes(title_text="actual RUL", row=1, col=2); fig.update_yaxes(title_text="predicted RUL", row=1, col=2)
fig.update_xaxes(title_text="error = predicted - actual (positive = late)", row=1, col=3); fig.update_yaxes(title_text="engines", row=1, col=3)
fig.update_layout(height=420, width=1250, title_text=f"{selected_name} on {DATASET_ID}: RMSE {test_scores['rmse']:.2f}, NASA {test_scores['nasa']:.0f}")
fig.show()

Validation-tail RMSE of every model, with the final test RMSE of the selected model for reference.

In [ ]:
# validation-tail RMSE of every model with the final test RMSE as a reference line
fig = go.Figure(go.Bar(x=list(comparison_table.index), y=comparison_table["tail RMSE"], marker_color=bar_colors, name="validation-tail RMSE"))
fig.add_hline(y=test_scores["rmse"], line_dash="dash", line_color="crimson", annotation_text="selected model, test RMSE")
fig.update_layout(height=380, width=800, yaxis_title="RMSE (cycles)", title_text="Validation-tail RMSE of all models (bars) and final test RMSE (line)")
fig.show()

Degradation view: the predicted RUL of the longest test engine at every cycle against the implied true RUL (`remaining life at the last cycle + cycles still to go`, capped like the training target). A good model follows the plateau and then declines with slope -1.

In [ ]:
longest_unit = int(final_test_prepared.groupby("unit").size().idxmax())
engine_rows = final_test_prepared[final_test_prepared["unit"] == longest_unit].reset_index(drop=True)
engine_windows, _, engine_cycles = build_windows(engine_rows, loaded_preprocessor["live_sensors"], WINDOW_SIZE)
engine_extra = engine_rows[final_extra_names].to_numpy()
engine_predictions = predict_with_loaded_model(engine_windows, engine_extra)
engine_true = np.minimum(test_true_rul[longest_unit - 1] + (engine_cycles.max() - engine_cycles), RUL_CAP)
fig = go.Figure()
fig.add_trace(go.Scatter(x=engine_cycles, y=engine_true, mode="lines", name="true RUL (capped)", line=dict(color="black")))
fig.add_trace(go.Scatter(x=engine_cycles, y=engine_predictions, mode="lines", name="predicted RUL", line=dict(color="crimson")))
fig.update_layout(height=380, width=800, xaxis_title="cycle", yaxis_title="RUL", title_text=f"Test engine {longest_unit}: RUL trajectory")
fig.show()

# 18. Final Project Verification
Executable checks of the guarantees this notebook makes. Every line either passes or stops the notebook.


## 18.1 Dataset Support and Data Separation

In [ ]:
checks = {}
checks["all four dataset ids are configured"] = set(EXPECTED_CONDITIONS) == {"FD001", "FD002", "FD003", "FD004"} and DATASET_ID in EXPECTED_CONDITIONS
checks["no engine is in both train and validation"] = set(train_part_df["unit"]).isdisjoint(set(val_part_df["unit"]))
checks["train + validation cover all training engines"] = len(set(train_part_df["unit"]) | set(val_part_df["unit"])) == train_df["unit"].nunique()
checks["test frame carries no RUL labels"] = not {"RUL", "RUL_linear"} & set(test_prepared_df.columns)
checks["validation scalers saw only training rows"] = fallback_scaler.n_samples_seen_ == len(train_part_df)
checks["final scalers saw only official training rows"] = final_fallback_scaler.n_samples_seen_ == len(train_df)
pd.Series(checks, name="passed").to_frame()

## 18.2 No Window Crosses an Engine Boundary
For every training engine, the last step of each window equals that engine's own row, and the first step of each full window equals the engine's row `window_size - 1` cycles earlier. A window that mixed two engines would break one of these equalities.

In [ ]:
boundary_ok = True
for unit_id in sorted(train_prepared_df["unit"].unique())[:25]:
    engine_values = train_prepared_df.loc[train_prepared_df["unit"] == unit_id, live_sensors].to_numpy(dtype=np.float32)
    engine_win = train_windows[train_window_units == unit_id]
    boundary_ok &= bool(np.allclose(engine_win[:, -1, :], engine_values))
    if len(engine_values) >= WINDOW_SIZE:
        boundary_ok &= bool(np.allclose(engine_win[WINDOW_SIZE - 1:, 0, :], engine_values[:len(engine_values) - WINDOW_SIZE + 1]))
    boundary_ok &= bool(np.allclose(engine_win[0], np.repeat(engine_values[:1], WINDOW_SIZE, axis=0)))
assert boundary_ok, "a window mixes observations of different engines"
print("window boundaries verified for 25 engines")

## 18.3 No Future Information in the Predictors
Rewrite every cycle after cycle *k* of one engine with random numbers and rebuild windows and features. Everything up to cycle *k* must be unchanged.

In [ ]:
probe_unit = int(train_prepared_df["unit"].iloc[0])
probe_rows = train_prepared_df[train_prepared_df["unit"] == probe_unit].reset_index(drop=True)
cut_cycle = len(probe_rows) // 2
altered_rows = probe_rows.copy()
altered_rows.loc[altered_rows["cycle"] > cut_cycle, live_sensors] = np.random.default_rng(0).normal(size=(int((altered_rows["cycle"] > cut_cycle).sum()), len(live_sensors)))
windows_a, _, cycles_a = build_windows(probe_rows, live_sensors, WINDOW_SIZE)
windows_b, _, _ = build_windows(altered_rows, live_sensors, WINDOW_SIZE)
features_a, _ = extract_window_features(windows_a, probe_rows[extra_names].to_numpy(), extra_names, live_sensors)
features_b, _ = extract_window_features(windows_b, altered_rows[extra_names].to_numpy(), extra_names, live_sensors)
past = cycles_a <= cut_cycle
assert np.allclose(windows_a[past], windows_b[past]) and np.allclose(features_a[past], features_b[past]), "past predictors changed when only the future was changed"
assert not np.allclose(features_a[~past], features_b[~past]), "the probe did not change the future rows"
print(f"features of cycles 1..{cut_cycle} are identical when later cycles change")

## 18.4 Final Predictions, Alignment, and Saved Models

In [ ]:
final_checks = {}
final_checks["one prediction per test engine"] = len(test_predictions) == test_raw_df["unit"].nunique() == len(test_rul_df)
final_checks["predictions are finite"] = bool(np.isfinite(test_predictions).all())
final_checks["metrics are finite"] = all(np.isfinite(v) for v in test_scores.values())
final_checks["test engines align with RUL file rows"] = bool(np.array_equal(inference_units, np.arange(1, len(test_rul_df) + 1)))
final_checks["model input shape matches windows"] = inference_windows.shape[1:] == (WINDOW_SIZE, len(loaded_preprocessor["live_sensors"]))
final_checks["loaded final model reproduces in-memory model"] = True
if selected_name in ("Random Forest", "XGBoost"):
    in_memory = np.clip(final_model.predict(final_X_test_df), 0.0, None)
else:
    in_memory = predict_network(final_model, torch.from_numpy(final_test_windows))
final_checks["loaded final model reproduces in-memory model"] = bool(np.allclose(in_memory, test_predictions, atol=1e-3))
for artifact_name in ("random_forest", "xgboost", "lstm", "cnn_lstm", "tcn", "temporal_transformer"):
    final_checks[f"artifact saved: {artifact_name}"] = any(ARTIFACT_PATH.glob(f"{artifact_name}_{DATASET_ID.lower()}*"))
final_checks["preprocessing artifacts saved"] = preprocessor_path.exists() and final_preprocessor_path.exists()
final_report = pd.Series(final_checks, name="passed").to_frame()
assert final_report["passed"].all(), f"verification failed:\n{final_report[~final_report['passed']]}"
final_report

## 18.5 Summary

In [ ]:
print(f"dataset {DATASET_ID} | selected model: {selected_name} (chosen on validation-tail RMSE {selected_result['tail']['rmse']:.2f})")
print(f"test RMSE {test_scores['rmse']:.2f} | MAE {test_scores['mae']:.2f} | NASA score {test_scores['nasa']:.1f}")
print("artifacts:", sorted(p.name for p in ARTIFACT_PATH.glob(f"*{DATASET_ID.lower()}*")))
print("all verification checks passed; the notebook ran from the first cell to the last")